In [1]:
# Purpose: Build the clean Pb(111) parent slab from the optimized bulk Pb calculation.
# Input: bulk/Pb_fcc/CONTCAR
# Output: surfaces/Pb111/clean/POSCAR
# Geometry: 3x3 surface cell, 4 Pb layers, 10 Å vacuum on each side.
# Constraints: bottom 2 Pb layers fixed; top 2 layers free.
# Notes: Uses the optimized FCC lattice constant directly from the bulk calculation.

from pathlib import Path

from ase.build import fcc111
from ase.constraints import FixAtoms
from ase.io import read, write

ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
BULK_PATH = ROOT / "bulk" / "Pb_fcc" / "CONTCAR"
OUT_DIR = ROOT / "surfaces" / "Pb111" / "clean"
OUT_PATH = OUT_DIR / "POSCAR"

bulk = read(BULK_PATH, format="vasp")
a = bulk.cell.lengths()[0]

slab = fcc111(
    "Pb",
    size=(3, 3, 4),
    a=a,
    vacuum=10.0,
    orthogonal=False,
)

tags = slab.get_tags()
slab.set_constraint(FixAtoms(mask=tags >= 3))

OUT_DIR.mkdir(parents=True, exist_ok=True)

if OUT_PATH.exists():
    raise FileExistsError(f"Refusing to overwrite existing structure: {OUT_PATH}")

write(OUT_PATH, slab, format="vasp", direct=True, vasp5=True)

fixed = [i for i, tag in enumerate(tags) if tag >= 3]

print(f"Bulk lattice constant: {a:.8f} Å")
print(f"Atoms: {len(slab)}")
print(f"PBC: {slab.pbc.tolist()}")
print(f"Layer tags: {sorted(set(tags))}")
print(f"Fixed atoms: {len(fixed)} / {len(slab)}")
print(f"Fixed indices: {fixed}")
print("Cell:")
print(slab.cell)
print(f"Wrote: {OUT_PATH}")

Bulk lattice constant: 4.97479496 Å
Atoms: 36
PBC: [True, True, False]
Layer tags: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Fixed atoms: 18 / 36
Fixed indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Cell:
Cell([[10.553133746283994, 0.0, 0.0], [5.276566873141997, 9.139281913816781, 0.0], [0.0, 0.0, 28.616597621913886]])
Wrote: /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/clean/POSCAR


In [2]:
# Purpose: Generate the first parallel batch of Pb(111) surface calculations.
#
# Input:
#   bulk/Pb_fcc/CONTCAR
#   _control/go.py
#   _control/submit_go.sh
#
# Outputs:
#   S01-S04: clean Pb(111) parent and numerical convergence structures
#   H01-H04: one-H adsorption-site screening structures
#   D01-D03: vacancy and Pb-adatom defect structures
#   G01-G03: clean Pb(111) structures at selected fixed electron chemical potentials
#
# Common model:
#   3x3 Pb(111) surface cell
#   Optimized bulk Pb lattice constant
#   15 Å cell space on each side of the slab
#   Bottom two Pb layers fixed
#   PBE+D3 / CANDLE + 0.5 M NaF via go.py
#
# Notes:
#   target-mu values correspond approximately to -1.0, -1.4, and -1.8 V vs RHE
#   at pH 7 under the potential convention used in the previous Pb calculations.
#   Directory names retain target-mu as the primary raw-calculation identifier.
#   This cell refuses to overwrite existing calculation directories.

from pathlib import Path
import json
import re

import numpy as np
from ase.build import add_adsorbate, fcc111
from ase.constraints import FixAtoms
from ase.io import read, write

ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
CONTROL = ROOT.parent / "_control"

BULK_PATH = ROOT / "bulk" / "Pb_fcc" / "CONTCAR"
GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

VACUUM_A = 15.0

bulk = read(BULK_PATH, format="vasp")
a = float(bulk.cell.lengths()[0])
d111 = a / np.sqrt(3.0)


def constrain_bottom_two(slab):
    tags = slab.get_tags()
    n_layers = int(tags.max())
    slab.set_constraint(FixAtoms(mask=tags >= n_layers - 1))
    return slab


def make_111(layers=4):
    slab = fcc111("Pb", size=(3, 3, layers), a=a, vacuum=VACUUM_A, orthogonal=False)
    return constrain_bottom_two(slab)


def setup_job(job_id, relpath, atoms, job_name, purpose, kpts=(4, 4, 1, "gamma"),
              target_mu=None, approx_U_RHE_V=None, maxstep=None,
              parent="bulk/Pb_fcc/CONTCAR"):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    job_dir.mkdir(parents=True)
    write(job_dir / "POSCAR", atoms, format="vasp", direct=True, vasp5=True)

    go_text = GO_TEMPLATE.read_text()
    submit_text = SUBMIT_TEMPLATE.read_text()

    kpt_text = f"({kpts[0]}, {kpts[1]}, {kpts[2]}, {kpts[3]!r})"
    go_text, n = re.subn(r"kpts=\([^)]*\)", f"kpts={kpt_text}", go_text, count=1)
    if n != 1:
        raise RuntimeError("Could not uniquely replace kpts in go.py")

    if target_mu is not None:
        marker = "fluid-anion F- 0.5\n"
        if marker not in go_text:
            raise RuntimeError("Could not locate electrolyte block in go.py")
        go_text = go_text.replace(marker, marker + f"target-mu {target_mu:.6f}\n", 1)

    if maxstep is not None:
        go_text, n = re.subn(r"maxstep\s*=\s*[0-9.]+", f"maxstep={maxstep}", go_text, count=1)
        if n != 1:
            raise RuntimeError("Could not uniquely replace FIRE maxstep in go.py")

    submit_text, n = re.subn(
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        submit_text,
        count=1,
        flags=re.MULTILINE,
    )
    if n != 1:
        raise RuntimeError("Could not uniquely replace Slurm job name")

    (job_dir / "go.py").write_text(go_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "purpose": purpose,
        "parent": parent,
        "bulk_lattice_constant_A": a,
        "vacuum_each_side_A": VACUUM_A,
        "kpts": list(kpts),
        "target_mu_Ha": target_mu,
        "approx_U_RHE_V_at_pH7": approx_U_RHE_V,
        "ase_pbc": [bool(x) for x in atoms.pbc],
        "n_atoms": len(atoms),
    }

    (job_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

    print(f"{job_id:>3}  {job_dir}")


# ---------------------------------------------------------------------
# S01-S04: clean Pb(111) and convergence
# ---------------------------------------------------------------------

setup_job(
    "S01", "surfaces/Pb111/clean", make_111(4), "Pb111_clean",
    "Primary clean Pb(111) parent: 4 layers, 4x4x1 k-point mesh.",
)

setup_job(
    "S02", "surfaces/Pb111/convergence/4L_k6", make_111(4), "Pb111_4L_k6",
    "Clean Pb(111), 4 layers, denser 6x6x1 k-point sampling.",
    kpts=(6, 6, 1, "gamma"),
)

setup_job(
    "S03", "surfaces/Pb111/convergence/5L_k4", make_111(5), "Pb111_5L_k4",
    "Clean Pb(111), 5 layers, slab-thickness comparison.",
)

setup_job(
    "S04", "surfaces/Pb111/convergence/6L_k4", make_111(6), "Pb111_6L_k4",
    "Clean Pb(111), 6 layers, slab-thickness comparison.",
)


# ---------------------------------------------------------------------
# H01-H04: one-H adsorption-site screening
# ---------------------------------------------------------------------

h_sites = {
    "H01": ("ontop", 1.8),
    "H02": ("bridge", 1.0),
    "H03": ("fcc", 1.0),
    "H04": ("hcp", 1.0),
}

for job_id, (site, height) in h_sites.items():
    slab = make_111(4)
    add_adsorbate(slab, "H", height=height, position=site, offset=(1, 1))

    setup_job(
        job_id,
        f"surfaces/Pb111/H_adsorption/{site}",
        slab,
        f"Pb111_H_{site}",
        f"One H initially adsorbed at the Pb(111) {site} site.",
        maxstep=0.10,
    )


# ---------------------------------------------------------------------
# D01-D03: simple corrosion/undercoordinated Pb models
# ---------------------------------------------------------------------

vacancy = make_111(4)
top_indices = np.where(vacancy.get_tags() == 1)[0]
vacancy_index = int(top_indices[len(top_indices) // 2])
del vacancy[vacancy_index]

setup_job(
    "D01", "surfaces/Pb111/defects/vacancy", vacancy, "Pb111_vac",
    "Pb(111) with one top-layer Pb vacancy; corrosion/extraction endpoint reference.",
)

for job_id, site in [("D02", "fcc"), ("D03", "hcp")]:
    slab = make_111(4)
    add_adsorbate(slab, "Pb", height=d111, position=site, offset=(1, 1))

    setup_job(
        job_id,
        f"surfaces/Pb111/defects/adatom_{site}",
        slab,
        f"Pb111_ad_{site}",
        f"Pb adatom initially placed at the Pb(111) {site} hollow.",
    )


# ---------------------------------------------------------------------
# G01-G03: clean Pb(111) under selected fixed electron chemical potentials
# ---------------------------------------------------------------------

fixed_mu_jobs = [
    ("G01", -0.1193, -1.0),
    ("G02", -0.1046, -1.4),
    ("G03", -0.0899, -1.8),
]

for job_id, mu, U_RHE in fixed_mu_jobs:
    mu_label = f"m{abs(mu):.4f}".replace(".", "p")

    setup_job(
        job_id,
        f"surfaces/Pb111/fixed_mu/{mu_label}",
        make_111(4),
        f"Pb111_{mu_label}",
        f"Clean Pb(111) relaxed at target-mu = {mu:.4f} Ha.",
        target_mu=mu,
        approx_U_RHE_V=U_RHE,
    )


print()
print(f"Bulk lattice constant used: {a:.8f} Å")
print(f"Vacuum / fluid-region spacing on each side: {VACUUM_A:.1f} Å")
print(f"Pb(111) interlayer spacing: {d111:.8f} Å")
print("Revised Pb(111) batch setup complete.")

S01  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/clean
S02  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/convergence/4L_k6
S03  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/convergence/5L_k4
S04  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/convergence/6L_k4
H01  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/H_adsorption/ontop
H02  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/H_adsorption/bridge
H03  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/H_adsorption/fcc
H04  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/H_adsorption/hcp
D01  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/defects/vacancy
D02  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/defects/adatom_fcc
D03  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/defects/adatom_hcp
G01  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/m0p1193
G02  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/m0p1046
G03  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/m0p0899

Bulk lattice constant used: 4.97479496 Å
Vacuum / fluid-regio

In [1]:
# Purpose: Generate Wave 2 Pb(111) hydrogen structures.
#
# Inputs:
#   bulk/Pb_fcc/CONTCAR
#   _control/go.py
#   _control/submit_go.sh
#
# Outputs:
#   SH01-SH03: candidate first-subsurface H structures
#   HH01-HH03: candidate two-H / PbH2-precursor structures
#
# Geometry:
#   3x3 Pb(111), 4 layers, 15 Å spacing on each side
#   bottom two Pb layers fixed
#
# Notes:
#   These calculations are intentionally generated from the common ideal Pb(111)
#   parent rather than from S01 so they can run in parallel with Wave 1.
#   Site labels describe INITIAL geometry only. Final relaxed structures must be classified afterward.

from pathlib import Path
import json
import re

import numpy as np
from ase import Atom
from ase.build import add_adsorbate, fcc111
from ase.constraints import FixAtoms
from ase.io import read, write
from ase.geometry import find_mic

ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
CONTROL = ROOT.parent / "_control"

BULK_PATH = ROOT / "bulk" / "Pb_fcc" / "CONTCAR"
GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

VACUUM_A = 15.0

bulk = read(BULK_PATH, format="vasp")
a = float(bulk.cell.lengths()[0])


def make_111():
    slab = fcc111("Pb", size=(3, 3, 4), a=a, vacuum=VACUUM_A, orthogonal=False)
    tags = slab.get_tags()
    slab.set_constraint(FixAtoms(mask=tags >= 3))
    return slab


def setup_job(job_id, relpath, atoms, job_name, purpose, maxstep=0.10):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    job_dir.mkdir(parents=True)
    write(job_dir / "POSCAR", atoms, format="vasp", direct=True, vasp5=True)

    go_text = GO_TEMPLATE.read_text()
    submit_text = SUBMIT_TEMPLATE.read_text()

    go_text, n = re.subn(r"maxstep\s*=\s*[0-9.]+", f"maxstep={maxstep}", go_text, count=1)
    if n != 1:
        raise RuntimeError("Could not uniquely replace FIRE maxstep in go.py")

    submit_text, n = re.subn(
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        submit_text,
        count=1,
        flags=re.MULTILINE,
    )
    if n != 1:
        raise RuntimeError("Could not uniquely replace Slurm job name")

    (job_dir / "go.py").write_text(go_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "purpose": purpose,
        "parent": "bulk/Pb_fcc/CONTCAR",
        "bulk_lattice_constant_A": a,
        "vacuum_each_side_A": VACUUM_A,
        "kpts": [4, 4, 1, "gamma"],
        "ase_pbc": [bool(x) for x in atoms.pbc],
        "n_atoms": len(atoms),
        "initial_geometry_only": True,
    }

    (job_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
    print(f"{job_id:>4}  {job_dir}")


def adsorption_xy(site):
    probe = make_111()
    add_adsorbate(probe, "H", height=1.0, position=site, offset=(1, 1))
    return probe.positions[-1, :2].copy()


# First and second Pb-layer z coordinates
reference = make_111()
tags = reference.get_tags()

z_top = float(reference.positions[tags == 1, 2].mean())
z_second = float(reference.positions[tags == 2, 2].mean())
z_sub = 0.5 * (z_top + z_second)

print(f"Top-layer z:       {z_top:.6f} Å")
print(f"Second-layer z:    {z_second:.6f} Å")
print(f"Subsurface H z0:   {z_sub:.6f} Å")


# ---------------------------------------------------------------------
# SH01-SH03: first-subsurface H candidate positions
# ---------------------------------------------------------------------

for job_id, site in [
    ("SH01", "fcc"),
    ("SH02", "hcp"),
    ("SH03", "ontop"),
]:
    slab = make_111()
    xy = adsorption_xy(site)

    slab += Atom("H", position=[xy[0], xy[1], z_sub])

    setup_job(
        job_id,
        f"surfaces/Pb111/H_subsurface/{site}",
        slab,
        f"Pb111_Hsub_{site}",
        f"One H initially placed halfway between the first and second Pb layers, laterally registered with the {site} surface site.",
        maxstep=0.08,
    )


# ---------------------------------------------------------------------
# HH01: ordinary two-H surface reference
# ---------------------------------------------------------------------

slab = make_111()
add_adsorbate(slab, "H", height=1.0, position="fcc", offset=(1, 1))
add_adsorbate(slab, "H", height=1.0, position="hcp", offset=(1, 1))

setup_job(
    "HH01",
    "surfaces/Pb111/H2_candidates/fcc_hcp",
    slab,
    "Pb111_2H_fcc_hcp",
    "Two surface H atoms initially occupying neighboring FCC and HCP hollow environments.",
)


# ---------------------------------------------------------------------
# HH02: two bridge H atoms sharing the same central surface Pb
# ---------------------------------------------------------------------

slab = make_111()
tags = slab.get_tags()
top_indices = np.where(tags == 1)[0]

center_xy = 0.5 * (slab.cell[0, :2] + slab.cell[1, :2])
center_index = min(top_indices, key=lambda i: np.linalg.norm(slab.positions[i, :2] - center_xy))

center = slab.positions[center_index].copy()

neighbor_data = []
for i in top_indices:
    if i == center_index:
        continue

    vec = slab.positions[i] - center
    mic_vec, dist = find_mic(vec, slab.cell, pbc=slab.pbc)
    neighbor_data.append((float(dist), i, mic_vec))

neighbor_data.sort(key=lambda x: x[0])

# Select two nearest neighbors that are not the same periodic direction.
n1 = neighbor_data[0]
n2 = next(item for item in neighbor_data[1:] if abs(np.dot(n1[2], item[2]) / (n1[0] * item[0])) < 0.9)

for _, _, vec in [n1, n2]:
    bridge = center + 0.5 * vec
    bridge[2] += 1.0
    slab += Atom("H", position=bridge)

setup_job(
    "HH02",
    "surfaces/Pb111/H2_candidates/shared_bridges",
    slab,
    "Pb111_2H_bridges",
    "Two H atoms initially placed at two bridge sites sharing the same top-layer Pb atom.",
)


# ---------------------------------------------------------------------
# HH03: direct PbH2-like local geometry
# ---------------------------------------------------------------------

slab = make_111()
tags = slab.get_tags()
top_indices = np.where(tags == 1)[0]

center_xy = 0.5 * (slab.cell[0, :2] + slab.cell[1, :2])
center_index = min(top_indices, key=lambda i: np.linalg.norm(slab.positions[i, :2] - center_xy))
pb = slab.positions[center_index].copy()

r_PbH = 1.90
theta = np.deg2rad(45.0)

dx = r_PbH * np.sin(theta)
dz = r_PbH * np.cos(theta)

slab += Atom("H", position=pb + [ dx, 0.0, dz])
slab += Atom("H", position=pb + [-dx, 0.0, dz])

setup_job(
    "HH03",
    "surfaces/Pb111/H2_candidates/PbH2_like",
    slab,
    "Pb111_PbH2_like",
    "Two H atoms initially arranged as a bent PbH2-like moiety on one top-layer Pb atom; Pb-H = 1.90 Å initial guess.",
    maxstep=0.06,
)

print()
print("Wave 2 setup complete.")

Top-layer z:       23.616598 Å
Second-layer z:    20.744398 Å
Subsurface H z0:   22.180498 Å
SH01  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/H_subsurface/fcc
SH02  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/H_subsurface/hcp
SH03  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/H_subsurface/ontop
HH01  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/H2_candidates/fcc_hcp
HH02  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/H2_candidates/shared_bridges
HH03  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/H2_candidates/PbH2_like

Wave 2 setup complete.


In [2]:
# Purpose: Generate fixed-potential Pb(111) H and 2H calculations.
#
# Inputs:
#   Existing H01-H04 and HH01-HH03 POSCAR structures
#   _control/go.py
#   _control/submit_go.sh
#
# Outputs:
#   12 one-H calculations: four adsorption geometries at three target-mu values
#    9 two-H calculations: three candidate geometries at three target-mu values
#
# Potentials:
#   target-mu = -0.1193 Ha  ~ -1.0 V vs RHE at pH 7
#   target-mu = -0.1046 Ha  ~ -1.4 V vs RHE at pH 7
#   target-mu = -0.0899 Ha  ~ -1.8 V vs RHE at pH 7
#
# Notes:
#   The target-mu value is the raw calculation condition.
#   Approximate RHE potentials are metadata only.
#   These geometries are independent children of the generated initial H/2H structures,
#   so they can run while the corresponding neutral relaxations are still queued.

from pathlib import Path
import json
import re

from ase.io import read, write

ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
CONTROL = ROOT.parent / "_control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

MU_STATES = [
    (-0.1193, -1.0),
    (-0.1046, -1.4),
    (-0.0899, -1.8),
]

H_PARENTS = {
    "ontop": ROOT / "surfaces/Pb111/H_adsorption/ontop/POSCAR",
    "bridge": ROOT / "surfaces/Pb111/H_adsorption/bridge/POSCAR",
    "fcc": ROOT / "surfaces/Pb111/H_adsorption/fcc/POSCAR",
    "hcp": ROOT / "surfaces/Pb111/H_adsorption/hcp/POSCAR",
}

H2_PARENTS = {
    "fcc_hcp": ROOT / "surfaces/Pb111/H2_candidates/fcc_hcp/POSCAR",
    "shared_bridges": ROOT / "surfaces/Pb111/H2_candidates/shared_bridges/POSCAR",
    "PbH2_like": ROOT / "surfaces/Pb111/H2_candidates/PbH2_like/POSCAR",
}


def mu_label(mu):
    return f"m{abs(mu):.4f}".replace(".", "p")


def setup_job(job_id, parent_path, relpath, job_name, purpose, mu, U_RHE, maxstep):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    atoms = read(parent_path, format="vasp")

    job_dir.mkdir(parents=True)
    write(job_dir / "POSCAR", atoms, format="vasp", direct=True, vasp5=True)

    go_text = GO_TEMPLATE.read_text()
    submit_text = SUBMIT_TEMPLATE.read_text()

    marker = "fluid-anion F- 0.5\n"
    if marker not in go_text:
        raise RuntimeError("Could not locate electrolyte block in go.py")

    go_text = go_text.replace(marker, marker + f"target-mu {mu:.6f}\n", 1)

    go_text, n = re.subn(
        r"maxstep\s*=\s*[0-9.]+",
        f"maxstep={maxstep}",
        go_text,
        count=1,
    )
    if n != 1:
        raise RuntimeError("Could not uniquely replace FIRE maxstep")

    submit_text, n = re.subn(
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        submit_text,
        count=1,
        flags=re.MULTILINE,
    )
    if n != 1:
        raise RuntimeError("Could not uniquely replace Slurm job name")

    (job_dir / "go.py").write_text(go_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "purpose": purpose,
        "parent_structure": str(parent_path.relative_to(ROOT)),
        "target_mu_Ha": mu,
        "approx_U_RHE_V_at_pH7": U_RHE,
        "kpts": [4, 4, 1, "gamma"],
        "n_atoms": len(atoms),
    }

    (job_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

    print(f"{job_id:>4}  {job_dir}")


# ---------------------------------------------------------------------
# One-H fixed-potential calculations
# MH01-MH12
# ---------------------------------------------------------------------

job_counter = 1

for site, parent in H_PARENTS.items():
    for mu, U_RHE in MU_STATES:
        mlab = mu_label(mu)
        job_id = f"MH{job_counter:02d}"

        setup_job(
            job_id=job_id,
            parent_path=parent,
            relpath=f"surfaces/Pb111/fixed_mu/H_adsorption/{site}/{mlab}",
            job_name=f"Pb111_H_{site}_{mlab}",
            purpose=f"One-H Pb(111) {site} geometry relaxed at target-mu = {mu:.4f} Ha.",
            mu=mu,
            U_RHE=U_RHE,
            maxstep=0.08,
        )

        job_counter += 1


# ---------------------------------------------------------------------
# Two-H / PbH2-like fixed-potential calculations
# M2H01-M2H09
# ---------------------------------------------------------------------

job_counter = 1

for geometry, parent in H2_PARENTS.items():
    for mu, U_RHE in MU_STATES:
        mlab = mu_label(mu)
        job_id = f"M2H{job_counter:02d}"

        setup_job(
            job_id=job_id,
            parent_path=parent,
            relpath=f"surfaces/Pb111/fixed_mu/H2_candidates/{geometry}/{mlab}",
            job_name=f"Pb111_2H_{geometry[:8]}_{mlab}",
            purpose=f"Two-H/PbH2 candidate '{geometry}' relaxed at target-mu = {mu:.4f} Ha.",
            mu=mu,
            U_RHE=U_RHE,
            maxstep=0.06,
        )

        job_counter += 1


print()
print("Wave 3 setup complete: 21 fixed-potential H/2H calculations.")

MH01  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/ontop/m0p1193
MH02  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/ontop/m0p1046
MH03  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/ontop/m0p0899
MH04  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/bridge/m0p1193
MH05  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/bridge/m0p1046
MH06  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/bridge/m0p0899
MH07  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/fcc/m0p1193
MH08  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/fcc/m0p1046
MH09  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/fcc/m0p0899
MH10  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/hcp/m0p1193
MH11  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_adsorption/hcp/m0p1046
MH12  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111

In [1]:
from pathlib import Path
import json
import re
import shutil

import numpy as np
from ase.geometry import find_mic
from ase.io import read, write


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
CONTROL = ROOT.parent / "_control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_GO_TEMPLATE = CONTROL / "submit_go.sh"
NEB_TEMPLATE = CONTROL / "neb.py"
SUBMIT_NEB_TEMPLATE = CONTROL / "submit_neb.sh"

TARGET_MU = -0.1193
NIMAGES = 8


def replace_once(text, pattern, replacement, description, flags=0):
    new_text, n = re.subn(pattern, replacement, text, count=1, flags=flags)

    if n != 1:
        raise RuntimeError(f"Could not uniquely replace {description}")

    return new_text


def surface_z(atoms):
    symbols = np.array(atoms.get_chemical_symbols())
    pb_indices = np.where(symbols == "Pb")[0]
    pb_z = atoms.positions[pb_indices, 2]

    return float(np.median(np.sort(pb_z)[-9:]))


def shared_pb_h2_indices(atoms):
    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    h_indices = np.where(symbols == "H")[0]

    if len(h_indices) != 2:
        raise ValueError(f"Expected exactly 2 H atoms, found {len(h_indices)}")

    nearest_pb = []

    for h_idx in h_indices:
        distances = np.array([
            atoms.get_distance(int(h_idx), int(pb_idx), mic=True)
            for pb_idx in pb_indices
        ])

        nearest_local = int(np.argmin(distances))
        nearest_pb.append(int(pb_indices[nearest_local]))

        if distances[nearest_local] > 2.6:
            raise ValueError(
                f"H atom {h_idx} is not clearly Pb-bound: nearest Pb-H = "
                f"{distances[nearest_local]:.3f} Å"
            )

    if nearest_pb[0] != nearest_pb[1]:
        raise ValueError(
            f"The two H atoms do not share one Pb: nearest Pb = {nearest_pb}"
        )

    return nearest_pb[0], [int(i) for i in h_indices]


def make_extraction_structure(parent_path, target_pb_lift_A):
    atoms = read(parent_path, format="vasp")

    pb_idx, h_indices = shared_pb_h2_indices(atoms)

    z_surface = surface_z(atoms)
    current_lift = float(atoms.positions[pb_idx, 2] - z_surface)
    dz = target_pb_lift_A - current_lift

    move_indices = [pb_idx, *h_indices]
    atoms.positions[move_indices, 2] += dz

    top_gap = float(atoms.cell[2, 2] - np.max(atoms.positions[:, 2]))

    if top_gap < 4.0:
        raise ValueError(
            f"Only {top_gap:.2f} Å remains above the highest atom. "
            "Increase the cell height before using this structure."
        )

    return atoms, {
        "shared_Pb_index": pb_idx,
        "H_indices": h_indices,
        "initial_Pb_lift_A": current_lift,
        "target_Pb_lift_A": target_pb_lift_A,
        "rigid_translation_A": dz,
        "top_gap_A": top_gap,
    }


def setup_relaxation(
    job_id,
    atoms,
    relpath,
    job_name,
    purpose,
    parent,
    target_mu=None,
    extra_metadata=None,
):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    job_dir.mkdir(parents=True)
    write(job_dir / "POSCAR", atoms, format="vasp", direct=True, vasp5=True)

    go_text = GO_TEMPLATE.read_text()
    submit_text = SUBMIT_GO_TEMPLATE.read_text()

    go_text = replace_once(
        go_text,
        r"maxstep\s*=\s*[0-9.]+",
        "maxstep=0.05",
        "FIRE maxstep",
    )

    if target_mu is not None:
        marker = "fluid-anion F- 0.5\n"

        if marker not in go_text:
            raise RuntimeError("Could not locate electrolyte block in go.py")

        go_text = go_text.replace(
            marker,
            marker + f"target-mu {target_mu:.6f}\n",
            1,
        )

    submit_text = replace_once(
        submit_text,
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        "Slurm job name",
        flags=re.MULTILINE,
    )

    (job_dir / "go.py").write_text(go_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "purpose": purpose,
        "parent": str(parent),
        "target_mu_Ha": target_mu,
        "kpts": [4, 4, 1, "gamma"],
    }

    if extra_metadata:
        metadata.update(extra_metadata)

    (job_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )

    print(f"{job_id:>5}  {job_dir}")


def copy_endpoint_outputs(source_dir, target_dir):
    for suffix in ("Ecomponents", "force"):
        candidates = sorted(
            source_dir.glob(f"*{suffix}"),
            key=lambda p: p.stat().st_mtime,
        )

        if not candidates:
            raise FileNotFoundError(
                f"No *{suffix} file found in endpoint directory {source_dir}"
            )

        source = candidates[-1]
        shutil.copy2(source, target_dir / source.name)


def interpolate_images(initial, final, nimages):
    if initial.get_chemical_symbols() != final.get_chemical_symbols():
        raise ValueError("Initial and final atom ordering/species differ")

    if not np.allclose(initial.cell.array, final.cell.array, atol=1e-8):
        raise ValueError("Initial and final cells differ")

    delta = final.positions - initial.positions
    pbc = np.array([True, True, False])

    for i in range(len(delta)):
        delta[i], _ = find_mic(delta[i], initial.cell, pbc=pbc)

    images = []

    for image_index in range(nimages + 2):
        fraction = image_index / (nimages + 1)

        image = initial.copy()
        image.positions = initial.positions + fraction * delta
        image.set_pbc((True, True, False))

        images.append(image)

    return images


def setup_neb(
    job_id,
    initial_dir,
    final_dir,
    relpath,
    job_name,
    purpose,
    target_mu=None,
):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    initial = read(initial_dir / "CONTCAR", format="vasp")
    final = read(final_dir / "CONTCAR", format="vasp")

    images = interpolate_images(initial, final, NIMAGES)

    job_dir.mkdir(parents=True)

    for idx, image in enumerate(images):
        image_dir = job_dir / f"{idx:02d}"
        image_dir.mkdir()

        if idx in (0, NIMAGES + 1):
            write(
                image_dir / "CONTCAR",
                image,
                format="vasp",
                direct=True,
                vasp5=True,
            )

            source_dir = initial_dir if idx == 0 else final_dir
            copy_endpoint_outputs(source_dir, image_dir)

        else:
            write(
                image_dir / "POSCAR",
                image,
                format="vasp",
                direct=True,
                vasp5=True,
            )

    neb_text = NEB_TEMPLATE.read_text()
    submit_text = SUBMIT_NEB_TEMPLATE.read_text()

    energy_label = "G" if target_mu is not None else "F"

    old_energy_function = re.compile(
        r"def _read_endpoint_energy\(s\):.*?"
        r"(?=\ndef _read_endpoint_forces)",
        flags=re.DOTALL,
    )

    new_energy_function = f'''def _read_endpoint_energy(s):
    f = next(n for n in os.listdir(s) if n.endswith('Ecomponents'))
    label = "{energy_label}"
    with open(os.path.join(s, f)) as fh:
        lines = [ln.strip() for ln in fh if ln.strip()]
    for line in reversed(lines):
        fields = line.split()
        if len(fields) >= 3 and fields[0] == label and fields[1] == '=':
            return float(fields[2]) * Hartree
    raise RuntimeError(f"Could not find {{label}} in {{os.path.join(s, f)}}")
'''

    neb_text, n = old_energy_function.subn(new_energy_function, neb_text, count=1)

    if n != 1:
        raise RuntimeError("Could not replace NEB endpoint-energy parser")

    if target_mu is not None:
        marker = "fluid-anion F- 0.5\n"

        if marker not in neb_text:
            raise RuntimeError("Could not locate electrolyte block in neb.py")

        neb_text = neb_text.replace(
            marker,
            marker + f"target-mu {target_mu:.6f}\n",
            1,
        )

    neb_text = replace_once(
        neb_text,
        r"maxstep\s*=\s*0\.2",
        "maxstep=0.05",
        "NEB FIRE maxstep",
    )

    neb_text = replace_once(
        neb_text,
        r"opt\.run\(fmax=0\.05,\s*steps=10\)",
        "opt.run(fmax=0.08, steps=150)",
        "NEB optimizer settings",
    )

    # First pass is deliberately non-climbing. CI-NEB comes after the band is stable.
    neb_text = replace_once(
        neb_text,
        r"climb=False",
        "climb=False",
        "NEB climb setting",
    )

    submit_text = replace_once(
        submit_text,
        r"^#SBATCH -q .*?$",
        "#SBATCH -q regular",
        "NEB QoS",
        flags=re.MULTILINE,
    )

    submit_text = replace_once(
        submit_text,
        r"^#SBATCH --time .*?$",
        "#SBATCH --time 08:00:00",
        "NEB walltime",
        flags=re.MULTILINE,
    )

    submit_text = replace_once(
        submit_text,
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        "NEB job name",
        flags=re.MULTILINE,
    )

    (job_dir / "neb.py").write_text(neb_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "purpose": purpose,
        "initial_endpoint": str(initial_dir.relative_to(ROOT)),
        "final_endpoint": str(final_dir.relative_to(ROOT)),
        "target_mu_Ha": target_mu,
        "endpoint_energy_label": energy_label,
        "n_intermediate_images": NIMAGES,
        "climb": False,
        "fmax_eV_A": 0.08,
        "maxstep_A": 0.05,
        "max_steps": 150,
        "stage": "initial_band_relaxation",
    }

    (job_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )

    print(f"{job_id:>5}  {job_dir}")


# ---------------------------------------------------------------------
# PbH2 extraction endpoint searches
# ---------------------------------------------------------------------

neutral_parent = (
    ROOT
    / "surfaces/Pb111/H2_candidates/shared_bridges/CONTCAR"
)

fixed_mu_parent = (
    ROOT
    / "surfaces/Pb111/fixed_mu/H2_candidates/PbH2_like/m0p1193/CONTCAR"
)

for i, target_lift in enumerate((2.5, 4.0, 6.0), start=1):
    atoms, info = make_extraction_structure(
        neutral_parent,
        target_pb_lift_A=target_lift,
    )

    label = str(target_lift).replace(".", "p")

    setup_relaxation(
        job_id=f"X{i:02d}",
        atoms=atoms,
        relpath=f"surfaces/Pb111/PbH2_extraction/neutral/Pb_lift_{label}",
        job_name=f"PbH2_N_z{label}",
        purpose=(
            f"Neutral PbH2-extraction endpoint search with the Pb(H)2 moiety "
            f"initially translated so its Pb atom is {target_lift:.1f} Å above "
            f"the median Pb(111) surface plane."
        ),
        parent=neutral_parent.relative_to(ROOT),
        target_mu=None,
        extra_metadata=info,
    )


for i, target_lift in enumerate((2.5, 4.0, 6.0), start=4):
    atoms, info = make_extraction_structure(
        fixed_mu_parent,
        target_pb_lift_A=target_lift,
    )

    label = str(target_lift).replace(".", "p")

    setup_relaxation(
        job_id=f"X{i:02d}",
        atoms=atoms,
        relpath=(
            f"surfaces/Pb111/PbH2_extraction/fixed_mu/"
            f"m0p1193/Pb_lift_{label}"
        ),
        job_name=f"PbH2_mu1193_z{label}",
        purpose=(
            f"Fixed-potential PbH2-extraction endpoint search at target-mu "
            f"{TARGET_MU:.4f} Ha with the Pb(H)2 moiety initially translated "
            f"so its Pb atom is {target_lift:.1f} Å above the median Pb(111) "
            f"surface plane."
        ),
        parent=fixed_mu_parent.relative_to(ROOT),
        target_mu=TARGET_MU,
        extra_metadata=info,
    )


# ---------------------------------------------------------------------
# Fixed-potential relaxation of the deep subsurface-H minimum
# ---------------------------------------------------------------------

subsurface_parent = (
    ROOT
    / "surfaces/Pb111/H_subsurface/ontop/CONTCAR"
)

subsurface_atoms = read(subsurface_parent, format="vasp")

setup_relaxation(
    job_id="X07",
    atoms=subsurface_atoms,
    relpath="surfaces/Pb111/fixed_mu/H_subsurface/deep/m0p1193",
    job_name="Pb111_Hsub_mu1193",
    purpose=(
        "Test whether the deep subsurface-H / Pb-lifted structure remains a "
        "local minimum at target-mu = -0.1193 Ha."
    ),
    parent=subsurface_parent.relative_to(ROOT),
    target_mu=TARGET_MU,
)


# ---------------------------------------------------------------------
# NEB setup
# ---------------------------------------------------------------------

setup_neb(
    job_id="N01",
    initial_dir=ROOT / "surfaces/Pb111/fixed_mu/H_adsorption/fcc/m0p1193",
    final_dir=ROOT / "surfaces/Pb111/fixed_mu/H_adsorption/ontop/m0p1193",
    relpath="kinetics/Pb111/fixed_mu/m0p1193/surface_H_to_lifted_PbH",
    job_name="NEB_H_lift_mu1193",
    purpose=(
        "Grand-canonical NEB from ordinary surface-bound H to the lifted Pb-H "
        "state at target-mu = -0.1193 Ha."
    ),
    target_mu=TARGET_MU,
)

setup_neb(
    job_id="N02",
    initial_dir=ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/fcc_hcp/m0p1193",
    final_dir=ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/shared_bridges/m0p1193",
    relpath="kinetics/Pb111/fixed_mu/m0p1193/separated_2H_to_same_Pb",
    job_name="NEB_2H_local_mu1193",
    purpose=(
        "Grand-canonical NEB from two H atoms associated with different Pb atoms "
        "to the same-Pb Pb(H)2 surface state at target-mu = -0.1193 Ha."
    ),
    target_mu=TARGET_MU,
)

setup_neb(
    job_id="N03",
    initial_dir=ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/shared_bridges/m0p1193",
    final_dir=ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/PbH2_like/m0p1193",
    relpath="kinetics/Pb111/fixed_mu/m0p1193/PbH2_surface_to_strongly_lifted",
    job_name="NEB_PbH2_lift_mu1193",
    purpose=(
        "Grand-canonical NEB from the mildly lifted same-Pb Pb(H)2 surface state "
        "to the strongly lifted incipient-extraction state at target-mu = -0.1193 Ha."
    ),
    target_mu=TARGET_MU,
)

setup_neb(
    job_id="N04",
    initial_dir=ROOT / "surfaces/Pb111/H_adsorption/fcc",
    final_dir=ROOT / "surfaces/Pb111/H_subsurface/ontop",
    relpath="kinetics/Pb111/neutral/surface_H_to_deep_subsurface_H",
    job_name="NEB_H_sub_neutral",
    purpose=(
        "Neutral NEB from the lowest ordinary surface-H state to the deep "
        "subsurface-H / Pb-lifted minimum."
    ),
    target_mu=None,
)

print()
print("Wave 4 setup complete: 7 relaxations + 4 NEBs.")

  X01  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/PbH2_extraction/neutral/Pb_lift_2p5
  X02  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/PbH2_extraction/neutral/Pb_lift_4p0
  X03  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/PbH2_extraction/neutral/Pb_lift_6p0
  X04  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/PbH2_extraction/fixed_mu/m0p1193/Pb_lift_2p5
  X05  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/PbH2_extraction/fixed_mu/m0p1193/Pb_lift_4p0
  X06  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/PbH2_extraction/fixed_mu/m0p1193/Pb_lift_6p0
  X07  /pscratch/sd/a/awasthi/Pb_LAR/surfaces/Pb111/fixed_mu/H_subsurface/deep/m0p1193
  N01  /pscratch/sd/a/awasthi/Pb_LAR/kinetics/Pb111/fixed_mu/m0p1193/surface_H_to_lifted_PbH
  N02  /pscratch/sd/a/awasthi/Pb_LAR/kinetics/Pb111/fixed_mu/m0p1193/separated_2H_to_same_Pb
  N03  /pscratch/sd/a/awasthi/Pb_LAR/kinetics/Pb111/fixed_mu/m0p1193/PbH2_surface_to_strongly_lifted
  N04  /pscratch/sd/a/awasthi/Pb_LAR/kinetics/Pb111/neutral/surface_H_t

In [1]:
from pathlib import Path
import json
import re

import numpy as np
from ase.io import read, write


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
CONTROL = ROOT.parent / "_control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

MANIFEST_DIR = ROOT / "analysis/manifests"
MANIFEST_PATH = MANIFEST_DIR / "wave5_PbH2_potential_sweep.json"

PRECURSOR_PARENT = (
    ROOT
    / "surfaces/Pb111/fixed_mu/H2_candidates/PbH2_like/m0p1193/CONTCAR"
)

DETACHED_PARENT = (
    ROOT
    / "surfaces/Pb111/PbH2_extraction/fixed_mu/m0p1193/Pb_lift_6p0/CONTCAR"
)

MU_POINTS = [
    ("X08", -0.11195, -1.2),
    ("X09", -0.10460, -1.4),
    ("X10", -0.09725, -1.6),
    ("X11", -0.08990, -1.8),
]

DETACHED_MU_POINTS = [
    ("X12", -0.11195, -1.2),
    ("X13", -0.10460, -1.4),
    ("X14", -0.09725, -1.6),
    ("X15", -0.08990, -1.8),
]


def replace_once(text, pattern, replacement, description, flags=0):
    new_text, n = re.subn(pattern, replacement, text, count=1, flags=flags)

    if n != 1:
        raise RuntimeError(f"Could not uniquely replace {description}")

    return new_text


def mu_label(mu):
    value = f"{abs(mu):.5f}".rstrip("0").rstrip(".").replace(".", "p")
    return f"m{value}" if mu < 0 else f"p{value}"


def setup_relaxation(
    job_id,
    atoms,
    relpath,
    job_name,
    purpose,
    source_job_id,
    source_path,
    target_mu,
    approx_U_RHE_V,
    state,
    extra_metadata=None,
):
    job_dir = ROOT / relpath

    if job_dir.exists():
        raise FileExistsError(f"Refusing to overwrite existing directory: {job_dir}")

    job_dir.mkdir(parents=True)

    write(
        job_dir / "POSCAR",
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
    )

    go_text = GO_TEMPLATE.read_text()
    submit_text = SUBMIT_TEMPLATE.read_text()

    if re.search(r"(?m)^\s*target-mu\s+", go_text):
        raise RuntimeError("The control go.py already contains target-mu")

    marker = "fluid-anion F- 0.5\n"

    if marker not in go_text:
        raise RuntimeError("Could not locate electrolyte block in control go.py")

    go_text = go_text.replace(
        marker,
        marker + f"target-mu {target_mu:.5f}\n",
        1,
    )

    go_text = replace_once(
        go_text,
        r"maxstep\s*=\s*[0-9.]+",
        "maxstep=0.05",
        "FIRE maxstep",
    )

    submit_text = replace_once(
        submit_text,
        r"^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        "Slurm job name",
        flags=re.MULTILINE,
    )

    (job_dir / "go.py").write_text(go_text)
    (job_dir / "submit.sh").write_text(submit_text)

    metadata = {
        "job_id": job_id,
        "family": "PbH2_potential_sweep",
        "state": state,
        "purpose": purpose,
        "source_job_id": source_job_id,
        "source_path": str(source_path.relative_to(ROOT)),
        "target_mu_Ha": target_mu,
        "approx_U_RHE_V_at_pH7": approx_U_RHE_V,
        "kpts": [4, 4, 1, "gamma"],
        "fmax_threshold_eV_A": 0.04,
        "fire_maxstep_A": 0.05,
    }

    if extra_metadata:
        metadata.update(extra_metadata)

    (job_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )

    return {
        "job_id": job_id,
        "directory": str(job_dir.relative_to(ROOT)),
        "state": state,
        "target_mu_Ha": target_mu,
        "approx_U_RHE_V_at_pH7": approx_U_RHE_V,
        "source_job_id": source_job_id,
        "purpose": purpose,
    }


def find_detached_pbh2(atoms):
    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    h_indices = np.where(symbols == "H")[0]

    if len(h_indices) != 2:
        raise ValueError(f"Expected exactly 2 H atoms, found {len(h_indices)}")

    nearest_pb = []
    nearest_distances = []

    for h_idx in h_indices:
        distances = np.array([
            atoms.get_distance(int(h_idx), int(pb_idx), mic=True)
            for pb_idx in pb_indices
        ])

        local = int(np.argmin(distances))

        nearest_pb.append(int(pb_indices[local]))
        nearest_distances.append(float(distances[local]))

    if nearest_pb[0] != nearest_pb[1]:
        raise ValueError(
            f"Two H atoms do not share one Pb: nearest Pb indices = {nearest_pb}"
        )

    if max(nearest_distances) > 2.4:
        raise ValueError(
            f"Expected intact PbH2; Pb-H distances = {nearest_distances}"
        )

    return nearest_pb[0], [int(i) for i in h_indices], nearest_distances


if MANIFEST_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing Wave 5 manifest: {MANIFEST_PATH}"
    )

if not PRECURSOR_PARENT.exists():
    raise FileNotFoundError(PRECURSOR_PARENT)

if not DETACHED_PARENT.exists():
    raise FileNotFoundError(DETACHED_PARENT)


precursor = read(PRECURSOR_PARENT, format="vasp")
detached = read(DETACHED_PARENT, format="vasp")

jobs = []


# ---------------------------------------------------------------------
# Strongly lifted surface Pb(H)2 precursor: potential sweep
# ---------------------------------------------------------------------

for job_id, mu, approx_U in MU_POINTS:
    label = mu_label(mu)

    jobs.append(
        setup_relaxation(
            job_id=job_id,
            atoms=precursor.copy(),
            relpath=(
                f"surfaces/Pb111/PbH2_extraction/potential_sweep/"
                f"precursor/{label}"
            ),
            job_name=f"PbH2_pre_{label}",
            purpose=(
                "Potential-dependent relaxation of the strongly lifted "
                "surface Pb(H)2 precursor."
            ),
            source_job_id="M2H07",
            source_path=PRECURSOR_PARENT,
            target_mu=mu,
            approx_U_RHE_V=approx_U,
            state="strongly_lifted_surface_PbH2",
        )
    )


# ---------------------------------------------------------------------
# Detached PbH2 + Pb vacancy: potential sweep
# ---------------------------------------------------------------------

for job_id, mu, approx_U in DETACHED_MU_POINTS:
    label = mu_label(mu)

    jobs.append(
        setup_relaxation(
            job_id=job_id,
            atoms=detached.copy(),
            relpath=(
                f"surfaces/Pb111/PbH2_extraction/potential_sweep/"
                f"detached/{label}"
            ),
            job_name=f"PbH2_det_{label}",
            purpose=(
                "Potential-dependent relaxation of detached PbH2 above "
                "a Pb(111) vacancy."
            ),
            source_job_id="X06",
            source_path=DETACHED_PARENT,
            target_mu=mu,
            approx_U_RHE_V=approx_U,
            state="detached_PbH2_plus_vacancy",
        )
    )


# ---------------------------------------------------------------------
# Separation-convergence check at target_mu = -0.1193 Ha
# ---------------------------------------------------------------------

farther = detached.copy()

pb_idx, h_indices, initial_pb_h = find_detached_pbh2(farther)

translation_A = 1.5
move_indices = [pb_idx, *h_indices]
farther.positions[move_indices, 2] += translation_A

top_gap_A = float(farther.cell[2, 2] - np.max(farther.positions[:, 2]))

if top_gap_A < 4.0:
    raise ValueError(
        f"Only {top_gap_A:.2f} A remains above the translated PbH2 unit; "
        "do not create X16 without increasing the cell height."
    )

jobs.append(
    setup_relaxation(
        job_id="X16",
        atoms=farther,
        relpath=(
            "surfaces/Pb111/PbH2_extraction/separation/"
            "fixed_mu/m0p1193/plus_1p5"
        ),
        job_name="PbH2_sep_mu1193",
        purpose=(
            "Separation-convergence check for detached PbH2 at "
            "target-mu = -0.1193 Ha; the intact PbH2 unit from X06 is "
            "translated 1.5 A farther from the Pb(111) surface."
        ),
        source_job_id="X06",
        source_path=DETACHED_PARENT,
        target_mu=-0.11930,
        approx_U_RHE_V=-1.0,
        state="detached_PbH2_farther_from_surface",
        extra_metadata={
            "Pb_index_translated": pb_idx,
            "H_indices_translated": h_indices,
            "initial_PbH_distances_A": initial_pb_h,
            "rigid_translation_z_A": translation_A,
            "remaining_top_gap_A": top_gap_A,
        },
    )
)


# ---------------------------------------------------------------------
# Immutable Wave 5 manifest
# ---------------------------------------------------------------------

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "wave": 5,
    "name": "PbH2 potential sweep and separation convergence",
    "scientific_question": (
        "Determine how the grand-free-energy difference between the "
        "strongly lifted surface Pb(H)2 precursor and detached PbH2 + "
        "vacancy changes with target electron chemical potential."
    ),
    "primary_variable": "target_mu_Ha",
    "potential_mapping_note": (
        "approx_U_RHE_V_at_pH7 uses the historical project calibration "
        "and is not the authoritative calculation variable."
    ),
    "jobs": jobs,
}

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n")

print(f"Created {len(jobs)} Wave 5 jobs.")
print(f"Manifest: {MANIFEST_PATH}")
print()

for job in jobs:
    print(
        f"{job['job_id']:>4}  "
        f"mu={job['target_mu_Ha']: .5f}  "
        f"{job['state']:<38}  "
        f"{job['directory']}"
    )

Created 9 Wave 5 jobs.
Manifest: /pscratch/sd/a/awasthi/Pb_LAR/analysis/manifests/wave5_PbH2_potential_sweep.json

 X08  mu=-0.11195  strongly_lifted_surface_PbH2            surfaces/Pb111/PbH2_extraction/potential_sweep/precursor/m0p11195
 X09  mu=-0.10460  strongly_lifted_surface_PbH2            surfaces/Pb111/PbH2_extraction/potential_sweep/precursor/m0p1046
 X10  mu=-0.09725  strongly_lifted_surface_PbH2            surfaces/Pb111/PbH2_extraction/potential_sweep/precursor/m0p09725
 X11  mu=-0.08990  strongly_lifted_surface_PbH2            surfaces/Pb111/PbH2_extraction/potential_sweep/precursor/m0p0899
 X12  mu=-0.11195  detached_PbH2_plus_vacancy              surfaces/Pb111/PbH2_extraction/potential_sweep/detached/m0p11195
 X13  mu=-0.10460  detached_PbH2_plus_vacancy              surfaces/Pb111/PbH2_extraction/potential_sweep/detached/m0p1046
 X14  mu=-0.09725  detached_PbH2_plus_vacancy              surfaces/Pb111/PbH2_extraction/potential_sweep/detached/m0p09725
 X15  mu=-0.0899

In [1]:
from pathlib import Path
import json
import re
import shutil
import subprocess
import sys

import numpy as np
from ase.io import read


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
CONTROL = ROOT.parent / "_control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_GO_TEMPLATE = CONTROL / "submit_go.sh"
NEB_TEMPLATE = CONTROL / "neb.py"
SUBMIT_NEB_TEMPLATE = CONTROL / "submit_neb.sh"

IDPP_SCRIPT = ROOT / "scripts/setup_neb_images.py"

MANIFEST_DIR = ROOT / "analysis/manifests"
MANIFEST_PATH = MANIFEST_DIR / "wave6_surface2H_and_extraction_nebs.json"

NIMAGES = 8


# ---------------------------------------------------------------------
# Existing endpoint / parent structures
# ---------------------------------------------------------------------

PATHS = {
    "M2H01": ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/fcc_hcp/m0p1193",
    "M2H04": ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/shared_bridges/m0p1193",
    "M2H05": ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/shared_bridges/m0p1046",
    "HH02": ROOT / "surfaces/Pb111/H2_candidates/shared_bridges",
    "X01": ROOT / "surfaces/Pb111/PbH2_extraction/neutral/Pb_lift_2p5",
    "X03": ROOT / "surfaces/Pb111/PbH2_extraction/neutral/Pb_lift_6p0",
    "M2H07": ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/PbH2_like/m0p1193",
    "X16": (
        ROOT
        / "surfaces/Pb111/PbH2_extraction/separation/"
        "fixed_mu/m0p1193/plus_1p5"
    ),
}


GO_JOBS = [
    {
        "job_id": "X17",
        "parent_id": "M2H01",
        "target_mu_Ha": -0.11195,
        "approx_U_RHE_V_at_pH7": -1.2,
        "state": "surface_2H_separated",
        "relpath": "surfaces/Pb111/fixed_mu/H2_reference/separated/m0p11195",
        "job_name": "H2sep_mu11195",
        "purpose": (
            "Test whether the separated two-H surface basin remains a "
            "physical local minimum at target-mu = -0.11195 Ha."
        ),
    },
    {
        "job_id": "X18",
        "parent_id": "M2H04",
        "target_mu_Ha": -0.11195,
        "approx_U_RHE_V_at_pH7": -1.2,
        "state": "surface_2H_same_Pb",
        "relpath": "surfaces/Pb111/fixed_mu/H2_reference/same_Pb/m0p11195",
        "job_name": "H2same_mu11195",
        "purpose": (
            "Test whether the same-Pb two-H surface basin remains a "
            "physical local minimum at target-mu = -0.11195 Ha."
        ),
    },
    {
        "job_id": "X19",
        "parent_id": "M2H05",
        "target_mu_Ha": -0.09725,
        "approx_U_RHE_V_at_pH7": -1.6,
        "state": "surface_2H_separated",
        "relpath": "surfaces/Pb111/fixed_mu/H2_reference/separated/m0p09725",
        "job_name": "H2sep_mu09725",
        "purpose": (
            "Continue a chemically valid separated two-H surface state "
            "to target-mu = -0.09725 Ha."
        ),
    },
    {
        "job_id": "X20",
        "parent_id": "M2H04",
        "target_mu_Ha": -0.09725,
        "approx_U_RHE_V_at_pH7": -1.6,
        "state": "surface_2H_same_Pb",
        "relpath": "surfaces/Pb111/fixed_mu/H2_reference/same_Pb/m0p09725",
        "job_name": "H2same_mu09725",
        "purpose": (
            "Test whether the same-Pb two-H surface basin survives at "
            "target-mu = -0.09725 Ha."
        ),
    },
]


NEB_JOBS = [
    {
        "job_id": "N05",
        "initial_id": "HH02",
        "final_id": "X01",
        "target_mu_Ha": None,
        "energy_key": "F",
        "relpath": "kinetics/Pb111/neutral/PbH2_surface_to_partially_extracted",
        "job_name": "NEB_PbH2_part_neut",
        "purpose": (
            "Neutral first-stage NEB from same-Pb surface Pb(H)2 to the "
            "partially extracted Pb(H)2 minimum."
        ),
    },
    {
        "job_id": "N06",
        "initial_id": "X01",
        "final_id": "X03",
        "target_mu_Ha": None,
        "energy_key": "F",
        "relpath": "kinetics/Pb111/neutral/PbH2_partially_extracted_to_detached",
        "job_name": "NEB_PbH2_det_neut",
        "purpose": (
            "Neutral first-stage NEB from partially extracted Pb(H)2 to "
            "detached PbH2 above a Pb vacancy."
        ),
    },
    {
        "job_id": "N07",
        "initial_id": "M2H07",
        "final_id": "X16",
        "target_mu_Ha": -0.11930,
        "energy_key": "G",
        "relpath": (
            "kinetics/Pb111/fixed_mu/m0p1193/"
            "PbH2_strongly_lifted_to_detached"
        ),
        "job_name": "NEB_PbH2_det_mu1193",
        "purpose": (
            "Fixed-mu first-stage NEB from strongly lifted surface Pb(H)2 "
            "to asymptotically detached PbH2 plus a Pb vacancy."
        ),
    },
]


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def replace_once(text, old, new, description):
    if text.count(old) != 1:
        raise RuntimeError(
            f"Expected exactly one occurrence of {description}; "
            f"found {text.count(old)}."
        )
    return text.replace(old, new, 1)


def latest_matching(directory, patterns):
    candidates = []

    for pattern in patterns:
        candidates.extend(directory.glob(pattern))

    candidates = [p for p in candidates if p.is_file()]

    if not candidates:
        raise FileNotFoundError(
            f"No file matching {patterns} found in {directory}"
        )

    return max(candidates, key=lambda p: p.stat().st_mtime)


def validate_endpoint_pair(initial_dir, final_dir):
    initial = read(initial_dir / "CONTCAR", format="vasp")
    final = read(final_dir / "CONTCAR", format="vasp")

    initial.set_pbc((True, True, False))
    final.set_pbc((True, True, False))

    if len(initial) != len(final):
        raise ValueError(
            f"Endpoint atom counts differ: {initial_dir} vs {final_dir}"
        )

    if initial.get_chemical_symbols() != final.get_chemical_symbols():
        raise ValueError(
            f"Endpoint atom species/order differ: {initial_dir} vs {final_dir}"
        )

    if not np.allclose(
        initial.cell.array,
        final.cell.array,
        atol=1e-8,
        rtol=0.0,
    ):
        raise ValueError(
            f"Endpoint cells differ: {initial_dir} vs {final_dir}"
        )


def make_go_text(template, target_mu):
    text = template

    marker = "fluid-anion F- 0.5\n"

    text = replace_once(
        text,
        marker,
        marker + f"target-mu {target_mu:.5f}\n",
        "fluid-anion command",
    )

    text = replace_once(
        text,
        "maxstep=0.2",
        "maxstep=0.05",
        "GO FIRE maxstep",
    )

    return text


def make_neb_text(template, energy_key, target_mu):
    text = template

    text = replace_once(
        text,
        "import os; from mpi4py import MPI",
        "import os; import re; from mpi4py import MPI",
        "NEB import line",
    )

    old_energy_reader = """def _read_endpoint_energy(s):
    f = next(n for n in os.listdir(s) if n.endswith('Ecomponents'))
    with open(os.path.join(s, f)) as fh: last = [ln.strip() for ln in fh if ln.strip()][-1]
    return float(last.split()[2]) * Hartree
"""

    new_energy_reader = f"""ENDPOINT_ENERGY_KEY = "{energy_key}"

def _read_endpoint_energy(s):
    pattern = re.compile(
        rf"^\\\\s*{{ENDPOINT_ENERGY_KEY}}\\\\s*=\\\\s*"
        r"([-+]?(?:\\\\d+(?:\\\\.\\\\d*)?|\\\\.\\\\d+)(?:[Ee][-+]?\\\\d+)?)"
    )
    values = []

    for name in sorted(os.listdir(s)):
        if not name.endswith("Ecomponents"):
            continue

        with open(os.path.join(s, name)) as fh:
            for line in fh:
                match = pattern.match(line)
                if match:
                    values.append(float(match.group(1)))

    if not values:
        raise RuntimeError(
            f"Could not find {{ENDPOINT_ENERGY_KEY}} in endpoint Ecomponents: {{s}}"
        )

    return values[-1] * Hartree
"""

    text = replace_once(
        text,
        old_energy_reader,
        new_energy_reader,
        "endpoint-energy reader",
    )

    if target_mu is not None:
        marker = "fluid-anion F- 0.5\n"

        text = replace_once(
            text,
            marker,
            marker + f"target-mu {target_mu:.5f}\n",
            "fluid-anion command",
        )

    text = replace_once(
        text,
        "maxstep=0.2",
        "maxstep=0.05",
        "NEB FIRE maxstep",
    )

    text = replace_once(
        text,
        "opt.run(fmax=0.05, steps=10)",
        "opt.run(fmax=0.08, steps=150)",
        "NEB first-stage convergence settings",
    )

    return text


def make_submit_text(template, job_name, neb=False):
    text = template

    text = re.sub(
        r"(?m)^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        text,
        count=1,
    )

    if neb:
        text = replace_once(
            text,
            "#SBATCH -q debug",
            "#SBATCH -q regular",
            "NEB QoS",
        )

        text = replace_once(
            text,
            "#SBATCH --time 00:30:00",
            "#SBATCH --time 08:00:00",
            "NEB walltime",
        )

    return text


def copy_endpoint_files(source_dir, destination_dir):
    destination_dir.mkdir(parents=True)

    shutil.copy2(
        source_dir / "CONTCAR",
        destination_dir / "CONTCAR",
    )

    ecomponents = latest_matching(
        source_dir,
        ["*Ecomponents", "*Ecomponents*"],
    )

    force = latest_matching(
        source_dir,
        ["*force", "*force*"],
    )

    shutil.copy2(
        ecomponents,
        destination_dir / "endpoint.Ecomponents",
    )

    shutil.copy2(
        force,
        destination_dir / "endpoint.force",
    )


# ---------------------------------------------------------------------
# Preflight: do not modify filesystem until everything checks out
# ---------------------------------------------------------------------

required_files = [
    GO_TEMPLATE,
    SUBMIT_GO_TEMPLATE,
    NEB_TEMPLATE,
    SUBMIT_NEB_TEMPLATE,
    IDPP_SCRIPT,
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(path)

for job_id, path in PATHS.items():
    if not (path / "CONTCAR").is_file():
        raise FileNotFoundError(
            f"{job_id} CONTCAR missing: {path / 'CONTCAR'}"
        )

for job in NEB_JOBS:
    initial_dir = PATHS[job["initial_id"]]
    final_dir = PATHS[job["final_id"]]

    validate_endpoint_pair(initial_dir, final_dir)

    latest_matching(initial_dir, ["*Ecomponents", "*Ecomponents*"])
    latest_matching(initial_dir, ["*force", "*force*"])
    latest_matching(final_dir, ["*Ecomponents", "*Ecomponents*"])
    latest_matching(final_dir, ["*force", "*force*"])

target_dirs = [
    ROOT / job["relpath"]
    for job in [*GO_JOBS, *NEB_JOBS]
]

existing = [path for path in target_dirs if path.exists()]

if existing:
    raise FileExistsError(
        "Refusing to overwrite existing Wave 6 directories:\n"
        + "\n".join(str(path) for path in existing)
    )

if MANIFEST_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing manifest: {MANIFEST_PATH}"
    )


# ---------------------------------------------------------------------
# Load templates
# ---------------------------------------------------------------------

go_template = GO_TEMPLATE.read_text()
submit_go_template = SUBMIT_GO_TEMPLATE.read_text()
neb_template = NEB_TEMPLATE.read_text()
submit_neb_template = SUBMIT_NEB_TEMPLATE.read_text()


# ---------------------------------------------------------------------
# Create X17-X20
# ---------------------------------------------------------------------

manifest_jobs = []

for job in GO_JOBS:
    job_dir = ROOT / job["relpath"]
    parent_dir = PATHS[job["parent_id"]]

    job_dir.mkdir(parents=True)

    shutil.copy2(
        parent_dir / "CONTCAR",
        job_dir / "POSCAR",
    )

    (job_dir / "go.py").write_text(
        make_go_text(
            go_template,
            job["target_mu_Ha"],
        )
    )

    (job_dir / "submit.sh").write_text(
        make_submit_text(
            submit_go_template,
            job["job_name"],
            neb=False,
        )
    )

    metadata = {
        "job_id": job["job_id"],
        "family": "fixed_mu_2H",
        "state": job["state"],
        "purpose": job["purpose"],
        "source_job_id": job["parent_id"],
        "source_path": str(parent_dir.relative_to(ROOT)),
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "kpts": [4, 4, 1, "gamma"],
        "fmax_threshold_eV_A": 0.04,
        "fire_maxstep_A": 0.05,
    }

    (job_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )

    manifest_jobs.append(
        {
            "job_id": job["job_id"],
            "type": "geometry_optimization",
            "directory": str(job_dir.relative_to(ROOT)),
            "source_job_id": job["parent_id"],
            "target_mu_Ha": job["target_mu_Ha"],
            "state": job["state"],
            "purpose": job["purpose"],
        }
    )


# ---------------------------------------------------------------------
# Create N05-N07
# ---------------------------------------------------------------------

for job in NEB_JOBS:
    neb_dir = ROOT / job["relpath"]
    initial_dir = PATHS[job["initial_id"]]
    final_dir = PATHS[job["final_id"]]

    neb_dir.mkdir(parents=True)

    copy_endpoint_files(
        initial_dir,
        neb_dir / "00",
    )

    for i in range(1, NIMAGES + 1):
        (neb_dir / f"{i:02d}").mkdir()

    copy_endpoint_files(
        final_dir,
        neb_dir / f"{NIMAGES + 1:02d}",
    )

    (neb_dir / "neb.py").write_text(
        make_neb_text(
            neb_template,
            energy_key=job["energy_key"],
            target_mu=job["target_mu_Ha"],
        )
    )

    (neb_dir / "submit.sh").write_text(
        make_submit_text(
            submit_neb_template,
            job["job_name"],
            neb=True,
        )
    )

    metadata = {
        "job_id": job["job_id"],
        "family": "NEB",
        "stage": "first_stage_nonclimbing",
        "purpose": job["purpose"],
        "initial_source_job_id": job["initial_id"],
        "initial_source_path": str(initial_dir.relative_to(ROOT)),
        "final_source_job_id": job["final_id"],
        "final_source_path": str(final_dir.relative_to(ROOT)),
        "target_mu_Ha": job["target_mu_Ha"],
        "endpoint_energy_key": job["energy_key"],
        "n_intermediate_images": NIMAGES,
        "climb": False,
        "neb_method": "improvedtangent",
        "fire_maxstep_A": 0.05,
        "fmax_threshold_eV_A": 0.08,
        "max_fire_steps": 150,
        "kpts": [4, 4, 1, "gamma"],
    }

    (neb_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )

    manifest_jobs.append(
        {
            "job_id": job["job_id"],
            "type": "NEB",
            "directory": str(neb_dir.relative_to(ROOT)),
            "initial_source_job_id": job["initial_id"],
            "final_source_job_id": job["final_id"],
            "target_mu_Ha": job["target_mu_Ha"],
            "endpoint_energy_key": job["energy_key"],
            "stage": "first_stage_nonclimbing",
            "purpose": job["purpose"],
        }
    )


# ---------------------------------------------------------------------
# Initialize N05-N07 with canonical MIC-aware IDPP
# ---------------------------------------------------------------------

print()
print("IDPP initialization")
print("===================")

for job in NEB_JOBS:
    neb_dir = ROOT / job["relpath"]

    print()
    print(f"{job['job_id']}: {neb_dir}")
    print("-" * 80)

    result = subprocess.run(
        [
            sys.executable,
            str(IDPP_SCRIPT),
            "--dir",
            str(neb_dir),
            "--nimages",
            str(NIMAGES),
        ],
        check=True,
        text=True,
        capture_output=True,
    )

    print(result.stdout)


# ---------------------------------------------------------------------
# Immutable Wave 6 manifest
# ---------------------------------------------------------------------

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "wave": 6,
    "name": "surface 2H potential references and PbH2 extraction NEBs",
    "scientific_questions": [
        (
            "Determine whether chemically valid surface two-H minima survive "
            "at target-mu = -0.11195 and -0.09725 Ha."
        ),
        (
            "Resolve the neutral Pb(H)2 extraction pathway into surface, "
            "partially extracted, and detached states."
        ),
        (
            "Calculate the first-stage grand-canonical PbH2 extraction band "
            "at target-mu = -0.1193 Ha."
        ),
    ],
    "potential_mapping_note": (
        "Approximate RHE potentials use the historical project calibration. "
        "target_mu_Ha is the authoritative calculation variable."
    ),
    "jobs": manifest_jobs,
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2) + "\n"
)

print()
print(f"Created {len(manifest_jobs)} Wave 6 jobs.")
print(f"Manifest: {MANIFEST_PATH}")
print()

for job in manifest_jobs:
    print(
        f"{job['job_id']:>4}  "
        f"{job['type']:<22}  "
        f"{job['directory']}"
    )


IDPP initialization

N05: /pscratch/sd/a/awasthi/Pb_LAR/kinetics/Pb111/neutral/PbH2_surface_to_partially_extracted
--------------------------------------------------------------------------------
NEB directory:       /pscratch/sd/a/awasthi/Pb_LAR/kinetics/Pb111/neutral/PbH2_surface_to_partially_extracted
Initial endpoint:    /pscratch/sd/a/awasthi/Pb_LAR/kinetics/Pb111/neutral/PbH2_surface_to_partially_extracted/00/CONTCAR
Final endpoint:      /pscratch/sd/a/awasthi/Pb_LAR/kinetics/Pb111/neutral/PbH2_surface_to_partially_extracted/09/CONTCAR
Intermediate images: 8
Interpolation:       IDPP, MIC=True

Maximum atomic displacement between neighboring images
-------------------------------------------------------
00 -> 01:   0.1934 A
01 -> 02:   0.1934 A
02 -> 03:   0.1934 A
03 -> 04:   0.1934 A
04 -> 05:   0.1934 A
05 -> 06:   0.1934 A
06 -> 07:   0.1934 A
07 -> 08:   0.1934 A
08 -> 09:   0.1934 A

Largest neighboring-image displacement: 0.1934 A
Intermediate POSCAR files written success

In [2]:
from pathlib import Path
import json
import re
import shutil

import numpy as np
from ase import Atoms
from ase.io import read, write


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
CONTROL = ROOT.parent / "_control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

PARENT_DIR = ROOT / "surfaces/Pb111/fixed_mu/H2_candidates/PbH2_like/m0p1193"
PARENT_PATH = PARENT_DIR / "CONTCAR"

MANIFEST_DIR = ROOT / "analysis/manifests"
MANIFEST_PATH = MANIFEST_DIR / "wave7_proton_well_rescue.json"

OH_BOND_A = 0.970
HOH_ANGLE_DEG = 104.5

# H_water ... H_Pb contact for water donating toward hydridic H
HYDRIDE_HH_CONTACT_A = 1.75

# O_water ... H_Pb contact for water accepting from proton-like H
PROTON_OH_CONTACT_A = 1.90

GO_MAXSTEP_A = 0.03


JOBS = [
    {
        "job_id": "X21",
        "orientation": "hydride_oriented",
        "target_mu_Ha": -0.11930,
        "approx_U_RHE_V_at_pH7": -1.0,
        "job_name": "PWres_hyd_m1193",
        "relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p1193",
    },
    {
        "job_id": "X22",
        "orientation": "hydride_oriented",
        "target_mu_Ha": -0.09725,
        "approx_U_RHE_V_at_pH7": -1.6,
        "job_name": "PWres_hyd_m09725",
        "relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p09725",
    },
    {
        "job_id": "X23",
        "orientation": "proton_oriented",
        "target_mu_Ha": -0.11930,
        "approx_U_RHE_V_at_pH7": -1.0,
        "job_name": "PWres_prot_m1193",
        "relpath": "surfaces/Pb111/proton_well_rescue/proton_oriented/m0p1193",
    },
    {
        "job_id": "X24",
        "orientation": "proton_oriented",
        "target_mu_Ha": -0.09725,
        "approx_U_RHE_V_at_pH7": -1.6,
        "job_name": "PWres_prot_m09725",
        "relpath": "surfaces/Pb111/proton_well_rescue/proton_oriented/m0p09725",
    },
]


def unit(vector):
    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)

    if norm < 1e-10:
        raise ValueError("Cannot normalize near-zero vector.")

    return vector / norm


def perpendicular_component(vector, axis):
    vector = np.asarray(vector, dtype=float)
    axis = unit(axis)
    result = vector - np.dot(vector, axis) * axis

    if np.linalg.norm(result) > 1e-6:
        return unit(result)

    # Robust fallback if the supplied reference happens to be parallel.
    candidates = [
        np.array([1.0, 0.0, 0.0]),
        np.array([0.0, 1.0, 0.0]),
        np.array([0.0, 0.0, 1.0]),
    ]

    candidate = min(candidates, key=lambda v: abs(np.dot(v, axis)))
    return unit(candidate - np.dot(candidate, axis) * axis)


def identify_pbh2(atoms):
    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    h_indices = np.where(symbols == "H")[0]

    if len(h_indices) != 2:
        raise ValueError(
            f"M2H07 parent should contain exactly 2 H; found {len(h_indices)}."
        )

    nearest_pb = []
    pb_h_distances = []

    for h_idx in h_indices:
        distances = np.array([
            atoms.get_distance(int(pb_idx), int(h_idx), mic=True)
            for pb_idx in pb_indices
        ])

        local = int(np.argmin(distances))
        nearest_pb.append(int(pb_indices[local]))
        pb_h_distances.append(float(distances[local]))

    if nearest_pb[0] != nearest_pb[1]:
        raise ValueError(
            f"The two parent H atoms do not share one Pb: {nearest_pb}"
        )

    if max(pb_h_distances) > 2.4:
        raise ValueError(
            f"Parent does not look like intact Pb(H)2: Pb-H = {pb_h_distances}"
        )

    return nearest_pb[0], [int(i) for i in h_indices], pb_h_distances


def local_geometry_vectors(atoms, pb_idx, h_indices, i):
    h_idx = h_indices[i]
    other_idx = h_indices[1 - i]

    # MIC vectors are used so this remains robust if the moiety sits near
    # a lateral periodic boundary.
    pb_to_h = atoms.get_distance(
        pb_idx,
        h_idx,
        mic=True,
        vector=True,
    )

    other_to_h = atoms.get_distance(
        other_idx,
        h_idx,
        mic=True,
        vector=True,
    )

    outward = unit(pb_to_h)

    # Pick an in-plane direction that points away from the other hydride.
    sideways = perpendicular_component(other_to_h, outward)

    return outward, sideways


def make_hydride_oriented_water(target_position, outward, sideways):
    """
    Construct H_Pb ... H_water-O-H.

    One water H is placed between the Pb-bound H and water O.
    The O-H bond therefore donates toward a hydride-like Pb-H.
    """
    theta = np.deg2rad(HOH_ANGLE_DEG)

    donor_h = target_position + HYDRIDE_HH_CONTACT_A * outward
    oxygen = donor_h + OH_BOND_A * outward

    # O -> donor H is -outward. Construct the second O-H vector at 104.5 deg.
    second_direction = (
        np.cos(theta) * (-outward)
        + np.sin(theta) * sideways
    )

    second_h = oxygen + OH_BOND_A * unit(second_direction)

    return Atoms(
        symbols=["O", "H", "H"],
        positions=[oxygen, donor_h, second_h],
    )


def make_proton_oriented_water(target_position, outward, sideways):
    """
    Construct H_Pb ... O(H)2.

    Water O points toward the Pb-bound H, with both water H atoms directed
    generally away from the target H.
    """
    half_angle = np.deg2rad(HOH_ANGLE_DEG / 2.0)

    oxygen = target_position + PROTON_OH_CONTACT_A * outward

    h1_direction = (
        np.cos(half_angle) * outward
        + np.sin(half_angle) * sideways
    )

    h2_direction = (
        np.cos(half_angle) * outward
        - np.sin(half_angle) * sideways
    )

    h1 = oxygen + OH_BOND_A * unit(h1_direction)
    h2 = oxygen + OH_BOND_A * unit(h2_direction)

    return Atoms(
        symbols=["O", "H", "H"],
        positions=[oxygen, h1, h2],
    )


def add_two_waters(parent, orientation, pb_idx, h_indices):
    atoms = parent.copy()

    water_records = []

    for i, h_idx in enumerate(h_indices):
        outward, sideways = local_geometry_vectors(
            parent,
            pb_idx,
            h_indices,
            i,
        )

        # Use the actual parent H coordinate as the construction anchor.
        target = parent.positions[h_idx].copy()

        if orientation == "hydride_oriented":
            water = make_hydride_oriented_water(
                target,
                outward,
                sideways,
            )

        elif orientation == "proton_oriented":
            water = make_proton_oriented_water(
                target,
                outward,
                sideways,
            )

        else:
            raise ValueError(f"Unknown orientation: {orientation}")

        water.set_cell(parent.cell)
        water.set_pbc(parent.pbc)

        first_new_index = len(atoms)
        atoms += water

        water_records.append(
            {
                "target_parent_H_index": h_idx,
                "water_initial_indices_before_go_sort": [
                    first_new_index,
                    first_new_index + 1,
                    first_new_index + 2,
                ],
            }
        )

    atoms.set_pbc((True, True, False))

    return atoms, water_records


def validate_rescue_geometry(atoms, pb_idx, parent_h_indices, orientation):
    symbols = np.array(atoms.get_chemical_symbols())

    n_pb = int(np.sum(symbols == "Pb"))
    n_o = int(np.sum(symbols == "O"))
    n_h = int(np.sum(symbols == "H"))

    if (n_pb, n_o, n_h) != (36, 2, 6):
        raise ValueError(
            f"Unexpected composition Pb/O/H = {n_pb}/{n_o}/{n_h}; "
            "expected 36/2/6."
        )

    # Added atoms are O,H,H,O,H,H before go.py performs its species sort.
    water_groups = [
        [38, 39, 40],
        [41, 42, 43],
    ]

    contact_distances = []

    for i, group in enumerate(water_groups):
        o_idx, wh1_idx, wh2_idx = group
        target_h = parent_h_indices[i]

        oh1 = atoms.get_distance(o_idx, wh1_idx, mic=True)
        oh2 = atoms.get_distance(o_idx, wh2_idx, mic=True)

        v1 = atoms.get_distance(o_idx, wh1_idx, mic=True, vector=True)
        v2 = atoms.get_distance(o_idx, wh2_idx, mic=True, vector=True)

        angle = np.degrees(
            np.arccos(
                np.clip(
                    np.dot(unit(v1), unit(v2)),
                    -1.0,
                    1.0,
                )
            )
        )

        if not (0.94 <= oh1 <= 1.00 and 0.94 <= oh2 <= 1.00):
            raise ValueError(
                f"Water {i + 1} O-H geometry is bad: {oh1:.3f}, {oh2:.3f} A"
            )

        if not (102.0 <= angle <= 107.0):
            raise ValueError(
                f"Water {i + 1} H-O-H angle is bad: {angle:.2f} deg"
            )

        if orientation == "hydride_oriented":
            contact = min(
                atoms.get_distance(target_h, wh1_idx, mic=True),
                atoms.get_distance(target_h, wh2_idx, mic=True),
            )
        else:
            contact = atoms.get_distance(target_h, o_idx, mic=True)

        contact_distances.append(float(contact))

    # Check different explicit waters are not accidentally overlapping.
    water1 = water_groups[0]
    water2 = water_groups[1]

    interwater_distances = [
        atoms.get_distance(i, j, mic=True)
        for i in water1
        for j in water2
    ]

    min_interwater = float(min(interwater_distances))

    if min_interwater < 1.35:
        raise ValueError(
            f"The two explicit waters overlap: minimum cross-water "
            f"distance = {min_interwater:.3f} A"
        )

    # Ensure enough z-space remains above every explicit atom.
    max_z = float(np.max(atoms.positions[:, 2]))
    top_gap = float(atoms.cell[2, 2] - max_z)

    if top_gap < 4.0:
        raise ValueError(
            f"Only {top_gap:.2f} A remains above explicit atoms."
        )

    pb_h = [
        atoms.get_distance(pb_idx, h_idx, mic=True)
        for h_idx in parent_h_indices
    ]

    hh = atoms.get_distance(
        parent_h_indices[0],
        parent_h_indices[1],
        mic=True,
    )

    return {
        "n_Pb": n_pb,
        "n_O": n_o,
        "n_H": n_h,
        "initial_PbH_distances_A": [float(x) for x in pb_h],
        "initial_HH_distance_A": float(hh),
        "initial_water_target_contacts_A": contact_distances,
        "initial_min_interwater_distance_A": min_interwater,
        "initial_top_gap_A": top_gap,
    }


def make_go_text(template, target_mu):
    text = template

    if re.search(r"(?m)^\s*target-mu\s+", text):
        raise RuntimeError(
            "Control go.py already contains target-mu; refusing ambiguous edit."
        )

    fluid_marker = "fluid-anion F- 0.5\n"

    if text.count(fluid_marker) != 1:
        raise RuntimeError(
            "Could not uniquely locate fluid-anion command in go.py."
        )

    text = text.replace(
        fluid_marker,
        fluid_marker + f"target-mu {target_mu:.5f}\n",
        1,
    )

    # Add a direct cavity diagnostic for this methodological campaign.
    cavity_marker = "dump Ionic BoundCharge VfluidTot FluidDensity\n"

    if text.count(cavity_marker) != 1:
        raise RuntimeError(
            "Could not uniquely locate fluid dump command in go.py."
        )

    text = text.replace(
        cavity_marker,
        cavity_marker + "dump Ionic Vcavity\n",
        1,
    )

    text, n = re.subn(
        r"maxstep\s*=\s*[0-9.]+",
        f"maxstep={GO_MAXSTEP_A:.2f}",
        text,
        count=1,
    )

    if n != 1:
        raise RuntimeError("Could not uniquely replace FIRE maxstep.")

    return text


def make_submit_text(template, job_name):
    text, n = re.subn(
        r"(?m)^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        template,
        count=1,
    )

    if n != 1:
        raise RuntimeError("Could not uniquely replace Slurm job name.")

    return text


# ---------------------------------------------------------------------
# Preflight
# ---------------------------------------------------------------------

for path in [GO_TEMPLATE, SUBMIT_TEMPLATE, PARENT_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

if MANIFEST_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing manifest: {MANIFEST_PATH}"
    )

target_dirs = [ROOT / job["relpath"] for job in JOBS]
existing = [path for path in target_dirs if path.exists()]

if existing:
    raise FileExistsError(
        "Refusing to overwrite existing Wave 7 directories:\n"
        + "\n".join(str(path) for path in existing)
    )


# ---------------------------------------------------------------------
# Parent Pb(H)2 identification
# ---------------------------------------------------------------------

parent = read(PARENT_PATH, format="vasp")
parent.set_pbc((True, True, False))

pb_idx, hydride_indices, parent_pb_h = identify_pbh2(parent)

print("Wave 7 parent")
print("=============")
print(f"Parent:           M2H07")
print(f"Common Pb index:  {pb_idx}")
print(f"Hydride indices:  {hydride_indices}")
print(
    "Pb-H distances:  "
    + ", ".join(f"{x:.4f} A" for x in parent_pb_h)
)
print(
    f"H-H distance:     "
    f"{parent.get_distance(hydride_indices[0], hydride_indices[1], mic=True):.4f} A"
)
print()


# ---------------------------------------------------------------------
# Create one starting structure per solvent orientation
# ---------------------------------------------------------------------

orientation_structures = {}
orientation_records = {}
orientation_metrics = {}

for orientation in ["hydride_oriented", "proton_oriented"]:
    atoms, water_records = add_two_waters(
        parent,
        orientation,
        pb_idx,
        hydride_indices,
    )

    metrics = validate_rescue_geometry(
        atoms,
        pb_idx,
        hydride_indices,
        orientation,
    )

    orientation_structures[orientation] = atoms
    orientation_records[orientation] = water_records
    orientation_metrics[orientation] = metrics


# ---------------------------------------------------------------------
# Create X21-X24
# ---------------------------------------------------------------------

go_template = GO_TEMPLATE.read_text()
submit_template = SUBMIT_TEMPLATE.read_text()

manifest_jobs = []

for job in JOBS:
    orientation = job["orientation"]
    atoms = orientation_structures[orientation].copy()
    metrics = orientation_metrics[orientation]

    job_dir = ROOT / job["relpath"]
    job_dir.mkdir(parents=True)

    write(
        job_dir / "POSCAR",
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
    )

    # Convenient human-readable preview / provenance copy.
    write(
        job_dir / "initial.xyz",
        atoms,
        format="extxyz",
    )

    (job_dir / "go.py").write_text(
        make_go_text(
            go_template,
            job["target_mu_Ha"],
        )
    )

    (job_dir / "submit.sh").write_text(
        make_submit_text(
            submit_template,
            job["job_name"],
        )
    )

    metadata = {
        "job_id": job["job_id"],
        "family": "proton_well_rescue",
        "state": "strongly_lifted_PbH2_plus_2H2O",
        "purpose": (
            "Test whether two explicit first-shell water molecules prevent "
            "the CANDLE proton-well artifact for strongly lifted surface "
            "Pb(H)2 at fixed electron chemical potential."
        ),
        "source_job_id": "M2H07",
        "source_path": str(PARENT_DIR.relative_to(ROOT)),
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "explicit_water_count": 2,
        "water_orientation": orientation,
        "construction": {
            "OH_bond_A": OH_BOND_A,
            "HOH_angle_deg": HOH_ANGLE_DEG,
            "hydride_Hwater_contact_A": (
                HYDRIDE_HH_CONTACT_A
                if orientation == "hydride_oriented"
                else None
            ),
            "proton_HOwater_contact_A": (
                PROTON_OH_CONTACT_A
                if orientation == "proton_oriented"
                else None
            ),
            "common_parent_Pb_index_before_go_sort": pb_idx,
            "parent_H_indices_before_go_sort": hydride_indices,
            "water_records_before_go_sort": orientation_records[orientation],
        },
        "initial_geometry_metrics": metrics,
        "kpts": [4, 4, 1, "gamma"],
        "fmax_threshold_eV_A": 0.04,
        "fire_maxstep_A": GO_MAXSTEP_A,
        "cavity_diagnostics": [
            "FluidDensity",
            "BoundCharge",
            "VfluidTot",
            "Vcavity",
        ],
        "potential_mapping_note": (
            "approx_U_RHE_V_at_pH7 uses the historical project calibration; "
            "target_mu_Ha is the authoritative calculation variable."
        ),
    }

    (job_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )

    manifest_jobs.append(
        {
            "job_id": job["job_id"],
            "directory": str(job_dir.relative_to(ROOT)),
            "orientation": orientation,
            "target_mu_Ha": job["target_mu_Ha"],
            "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
            "source_job_id": "M2H07",
        }
    )


# ---------------------------------------------------------------------
# Wave 7 manifest
# ---------------------------------------------------------------------

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "wave": 7,
    "name": "explicit-water proton-well rescue pilot",
    "scientific_question": (
        "Can a chemically bound strongly lifted Pb(H)2 state survive at "
        "cathodic target-mu when two explicit first-shell water molecules "
        "prevent the implicit-solvent cavity from approaching bare H directly?"
    ),
    "strategy": (
        "Compare hydride-oriented and proton-oriented water motifs at a "
        "known-good control potential (-0.11930 Ha) and a potential where "
        "dry CANDLE calculations exhibit the proton-well artifact "
        "(-0.09725 Ha)."
    ),
    "jobs": manifest_jobs,
}

MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2) + "\n"
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Wave 7 starting-geometry checks")
print("===============================")

for orientation in ["hydride_oriented", "proton_oriented"]:
    m = orientation_metrics[orientation]

    print()
    print(orientation)
    print("-" * len(orientation))
    print(
        "Pb-H:             "
        + ", ".join(f"{x:.4f} A" for x in m["initial_PbH_distances_A"])
    )
    print(f"PbH2 H-H:          {m['initial_HH_distance_A']:.4f} A")
    print(
        "water contacts:   "
        + ", ".join(
            f"{x:.4f} A"
            for x in m["initial_water_target_contacts_A"]
        )
    )
    print(
        f"min water-water:   "
        f"{m['initial_min_interwater_distance_A']:.4f} A"
    )
    print(f"top vacuum gap:    {m['initial_top_gap_A']:.4f} A")

print()
print(f"Created {len(JOBS)} Wave 7 jobs.")
print(f"Manifest: {MANIFEST_PATH}")
print()

for job in manifest_jobs:
    print(
        f"{job['job_id']:>4}  "
        f"mu={job['target_mu_Ha']: .5f}  "
        f"{job['orientation']:<18}  "
        f"{job['directory']}"
    )

Wave 7 parent
Parent:           M2H07
Common Pb index:  32
Hydride indices:  [36, 37]
Pb-H distances:  2.0233 A, 2.0233 A
H-H distance:     2.6153 A

Wave 7 starting-geometry checks

hydride_oriented
----------------
Pb-H:             2.0233 A, 2.0233 A
PbH2 H-H:          2.6153 A
water contacts:   1.7500 A, 1.7500 A
min water-water:   2.6749 A
top vacuum gap:    10.2789 A

proton_oriented
---------------
Pb-H:             2.0233 A, 2.0233 A
PbH2 H-H:          2.6153 A
water contacts:   1.9000 A, 1.9000 A
min water-water:   3.5438 A
top vacuum gap:    9.9559 A

Created 4 Wave 7 jobs.
Manifest: /pscratch/sd/a/awasthi/Pb_LAR/analysis/manifests/wave7_proton_well_rescue.json

 X21  mu=-0.11930  hydride_oriented    surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p1193
 X22  mu=-0.09725  hydride_oriented    surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p09725
 X23  mu=-0.11930  proton_oriented     surfaces/Pb111/proton_well_rescue/proton_oriented/m0p1193
 X24  mu=-0.09725  proton

In [3]:
# Wave 8: continue the successful hydride-oriented Pb(H)2 + 2H2O branch across potential.
#
# Inputs:
#   X21 CONTCAR at target_mu = -0.11930 Ha
#   X22 CONTCAR at target_mu = -0.09725 Ha
#
# Outputs:
#   X25: -0.11195 Ha
#   X26: -0.10460 Ha
#   X27: -0.08990 Ha
#   analysis/manifests/wave8_explicit_water_potential_continuation.json
#
# Assumptions:
#   - X21 and X22 are converged physical hydride-oriented solvated Pb(H)2 states.
#   - Their local go.py files already contain the desired Wave 7 settings:
#       PBE+D3, CANDLE, fixed mu, FIRE maxstep = 0.03 A, and Vcavity diagnostics.
#   - target_mu_Ha is authoritative; U_RHE values are historical approximate mappings.
#   - Existing directories are never overwritten.

from pathlib import Path
import csv
import json
import re

from ase.io import read, write


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
RESULTS_CSV = ROOT / "analysis/surface_results.csv"

MANIFEST_PATH = ROOT / "analysis/manifests/wave8_explicit_water_potential_continuation.json"


JOBS = [
    {
        "job_id": "X25",
        "source_job_id": "X21",
        "source_relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p1193",
        "target_mu_Ha": -0.11195,
        "approx_U_RHE_V_at_pH7": -1.2,
        "job_name": "PWres_hyd_m11195",
        "relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p11195",
    },
    {
        "job_id": "X26",
        "source_job_id": "X22",
        "source_relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p09725",
        "target_mu_Ha": -0.10460,
        "approx_U_RHE_V_at_pH7": -1.4,
        "job_name": "PWres_hyd_m1046",
        "relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p1046",
    },
    {
        "job_id": "X27",
        "source_job_id": "X22",
        "source_relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p09725",
        "target_mu_Ha": -0.08990,
        "approx_U_RHE_V_at_pH7": -1.8,
        "job_name": "PWres_hyd_m0899",
        "relpath": "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p0899",
    },
]


def load_harvest_status(csv_path):
    if not csv_path.is_file():
        raise FileNotFoundError(
            f"Missing harvested results: {csv_path}\n"
            "Run harvest_surface_results.py first."
        )

    rows = {}

    with csv_path.open(newline="") as handle:
        for row in csv.DictReader(handle):
            rows[row["job_id"]] = row

    return rows


def replace_target_mu(text, target_mu):
    pattern = r"(?m)^\s*target-mu\s+[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[Ee][-+]?\d+)?\s*$"

    text, n = re.subn(
        pattern,
        f"target-mu {target_mu:.5f}",
        text,
        count=1,
    )

    if n != 1:
        raise RuntimeError(
            f"Expected exactly one target-mu command; replaced {n}."
        )

    return text


def replace_job_name(text, job_name):
    text, n = re.subn(
        r"(?m)^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        text,
        count=1,
    )

    if n != 1:
        raise RuntimeError(
            f"Expected exactly one Slurm job-name line; replaced {n}."
        )

    return text


# ---------------------------------------------------------------------
# Preflight
# ---------------------------------------------------------------------

if MANIFEST_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing manifest: {MANIFEST_PATH}"
    )

harvest = load_harvest_status(RESULTS_CSV)

for job in JOBS:
    source_job_id = job["source_job_id"]

    if source_job_id not in harvest:
        raise KeyError(f"{source_job_id} not found in {RESULTS_CSV}")

    status = harvest[source_job_id]["status"]

    if status != "CONVERGED":
        raise RuntimeError(
            f"{source_job_id} is not converged: status={status}"
        )

    source_dir = ROOT / job["source_relpath"]
    target_dir = ROOT / job["relpath"]

    for required in ["CONTCAR", "go.py", "submit.sh", "metadata.json"]:
        path = source_dir / required

        if not path.is_file():
            raise FileNotFoundError(path)

    if target_dir.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing directory: {target_dir}"
        )


# ---------------------------------------------------------------------
# Create jobs
# ---------------------------------------------------------------------

manifest_jobs = []

for job in JOBS:
    source_dir = ROOT / job["source_relpath"]
    target_dir = ROOT / job["relpath"]

    atoms = read(source_dir / "CONTCAR", format="vasp")
    atoms.set_pbc((True, True, False))

    symbols = atoms.get_chemical_symbols()

    if symbols.count("Pb") != 36 or symbols.count("O") != 2 or symbols.count("H") != 6:
        raise ValueError(
            f"{job['source_job_id']} has unexpected composition: "
            f"Pb={symbols.count('Pb')}, O={symbols.count('O')}, H={symbols.count('H')}"
        )

    source_go = (source_dir / "go.py").read_text()

    if "Vcavity" not in source_go:
        raise RuntimeError(
            f"{job['source_job_id']} go.py does not contain Vcavity diagnostics."
        )

    source_submit = (source_dir / "submit.sh").read_text()

    target_dir.mkdir(parents=True)

    write(
        target_dir / "POSCAR",
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
    )

    write(
        target_dir / "initial.xyz",
        atoms,
        format="extxyz",
    )

    (target_dir / "go.py").write_text(
        replace_target_mu(
            source_go,
            job["target_mu_Ha"],
        )
    )

    (target_dir / "submit.sh").write_text(
        replace_job_name(
            source_submit,
            job["job_name"],
        )
    )

    metadata = {
        "job_id": job["job_id"],
        "family": "proton_well_rescue",
        "state": "strongly_lifted_PbH2_plus_2H2O",
        "purpose": (
            "Continue the successful hydride-oriented explicit-water Pb(H)2 "
            "branch across fixed electron chemical potential to test whether "
            "first-shell water suppresses the CANDLE proton-well artifact."
        ),
        "source_job_id": job["source_job_id"],
        "source_path": job["source_relpath"],
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "explicit_water_count": 2,
        "water_orientation": "hydride_oriented",
        "continuation_strategy": (
            "Use the CONTCAR from the nearest available converged "
            "hydride-oriented explicit-water state."
        ),
        "cavity_diagnostics": [
            "FluidDensity",
            "BoundCharge",
            "VfluidTot",
            "Vcavity",
        ],
        "potential_mapping_note": (
            "approx_U_RHE_V_at_pH7 uses the historical project calibration; "
            "target_mu_Ha is the authoritative calculation variable."
        ),
    }

    (target_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )

    manifest_jobs.append(
        {
            "job_id": job["job_id"],
            "directory": job["relpath"],
            "source_job_id": job["source_job_id"],
            "target_mu_Ha": job["target_mu_Ha"],
            "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        }
    )


# ---------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------

manifest = {
    "wave": 8,
    "name": "hydride-oriented explicit-water potential continuation",
    "scientific_question": (
        "Does the solvated strongly lifted Pb(H)2 local minimum persist "
        "through the cathodic potential range where dry CANDLE calculations "
        "fall into the proton well?"
    ),
    "reference_states": {
        "X21": {
            "target_mu_Ha": -0.11930,
            "role": "converged explicit-water control",
        },
        "X22": {
            "target_mu_Ha": -0.09725,
            "role": "converged explicit-water rescue in proton-well regime",
        },
    },
    "jobs": manifest_jobs,
}

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2) + "\n"
)


# ---------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------

print("Wave 8 created")
print("==============")
print(f"Manifest: {MANIFEST_PATH}")
print()

for job in manifest_jobs:
    print(
        f"{job['job_id']:>4}  "
        f"mu={job['target_mu_Ha']: .5f}  "
        f"source={job['source_job_id']:<3}  "
        f"{job['directory']}"
    )

Wave 8 created
Manifest: /pscratch/sd/a/awasthi/Pb_LAR/analysis/manifests/wave8_explicit_water_potential_continuation.json

 X25  mu=-0.11195  source=X21  surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p11195
 X26  mu=-0.10460  source=X22  surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p1046
 X27  mu=-0.08990  source=X22  surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p0899


In [4]:
# Create a persistent water-aware geometry analyzer for explicit-water Pb-H jobs.
#
# Classification:
#   - H with nearest O <= 1.25 A -> water-associated H
#   - remaining H -> non-water H, analyzed against Pb
#
# Output:
#   analysis/explicit_water_states.csv
#
# This does not replace the general Pb/H classifier; it handles jobs containing explicit H2O.

from pathlib import Path


SCRIPT_PATH = Path(
    "/pscratch/sd/a/awasthi/Pb_LAR/analysis/analyze_explicit_water_states.py"
)

if SCRIPT_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing analyzer: {SCRIPT_PATH}"
    )


SCRIPT = r'''from pathlib import Path
import csv
import json

import numpy as np
from ase.io import read


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
SURFACE_RESULTS = ROOT / "analysis/surface_results.csv"
OUTPUT = ROOT / "analysis/explicit_water_states.csv"

OH_COVALENT_MAX_A = 1.25
PROTON_WELL_PBH_MIN_A = 3.0
H2_MAX_A = 0.95


def unit(v):
    v = np.asarray(v, dtype=float)
    return v / np.linalg.norm(v)


def angle_deg(v1, v2):
    cosine = np.clip(np.dot(unit(v1), unit(v2)), -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))


def load_harvest():
    if not SURFACE_RESULTS.is_file():
        return {}

    rows = {}

    with SURFACE_RESULTS.open(newline="") as handle:
        for row in csv.DictReader(handle):
            rows[row["job_id"]] = row

    return rows


def surface_plane_z(atoms, pb_indices):
    z = np.array([atoms.positions[i, 2] for i in pb_indices])
    return float(np.median(np.sort(z)[-9:]))


def analyze_job(job_dir, metadata, harvest):
    structure_path = job_dir / "CONTCAR"

    if not structure_path.is_file():
        return None

    atoms = read(structure_path, format="vasp")
    atoms.set_pbc((True, True, False))

    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    o_indices = np.where(symbols == "O")[0]
    h_indices = np.where(symbols == "H")[0]

    if len(o_indices) == 0:
        return None

    surface_z = surface_plane_z(atoms, pb_indices)

    nearest_o = {}

    for h_idx in h_indices:
        distances = [
            atoms.get_distance(int(h_idx), int(o_idx), mic=True)
            for o_idx in o_indices
        ]

        local = int(np.argmin(distances))

        nearest_o[int(h_idx)] = {
            "o_idx": int(o_indices[local]),
            "distance_A": float(distances[local]),
        }

    water_h = [
        int(h_idx)
        for h_idx in h_indices
        if nearest_o[int(h_idx)]["distance_A"] <= OH_COVALENT_MAX_A
    ]

    nonwater_h = [
        int(h_idx)
        for h_idx in h_indices
        if int(h_idx) not in water_h
    ]

    water_counts = []
    oh_distances = []
    hoh_angles = []

    for o_idx in o_indices:
        assigned = [
            h_idx
            for h_idx in water_h
            if nearest_o[h_idx]["o_idx"] == int(o_idx)
        ]

        water_counts.append(len(assigned))

        for h_idx in assigned:
            oh_distances.append(
                atoms.get_distance(int(o_idx), h_idx, mic=True)
            )

        if len(assigned) == 2:
            v1 = atoms.get_distance(
                int(o_idx),
                assigned[0],
                mic=True,
                vector=True,
            )

            v2 = atoms.get_distance(
                int(o_idx),
                assigned[1],
                mic=True,
                vector=True,
            )

            hoh_angles.append(angle_deg(v1, v2))

    pb_h_distances = []
    pb_h_zrel = []
    nearest_pb_indices = []

    for h_idx in nonwater_h:
        distances = [
            atoms.get_distance(h_idx, int(pb_idx), mic=True)
            for pb_idx in pb_indices
        ]

        local = int(np.argmin(distances))

        pb_h_distances.append(float(distances[local]))
        nearest_pb_indices.append(int(pb_indices[local]))
        pb_h_zrel.append(float(atoms.positions[h_idx, 2] - surface_z))

    hh_distances = []

    for i, h1 in enumerate(h_indices):
        for h2 in h_indices[i + 1:]:
            hh_distances.append(
                atoms.get_distance(int(h1), int(h2), mic=True)
            )

    nonwater_to_water_h = []

    for h1 in nonwater_h:
        for h2 in water_h:
            nonwater_to_water_h.append(
                atoms.get_distance(h1, h2, mic=True)
            )

    nonwater_to_o = []

    for h_idx in nonwater_h:
        for o_idx in o_indices:
            nonwater_to_o.append(
                atoms.get_distance(h_idx, int(o_idx), mic=True)
            )

    harvest_row = harvest.get(metadata["job_id"], {})

    proton_well_like = any(
        distance > PROTON_WELL_PBH_MIN_A
        for distance in pb_h_distances
    )

    row = {
        "job_id": metadata["job_id"],
        "status": harvest_row.get("status", ""),
        "final_fmax_eV_A": harvest_row.get("final_fmax_eV_A", ""),
        "target_mu_Ha": metadata.get("target_mu_Ha", ""),
        "water_orientation": metadata.get("water_orientation", ""),
        "n_Pb": len(pb_indices),
        "n_O": len(o_indices),
        "n_H_total": len(h_indices),
        "n_water_H": len(water_h),
        "n_nonwater_H": len(nonwater_h),
        "water_H_counts_per_O": ";".join(str(x) for x in water_counts),
        "water_intact": all(x == 2 for x in water_counts),
        "OH_min_A": min(oh_distances) if oh_distances else "",
        "OH_max_A": max(oh_distances) if oh_distances else "",
        "HOH_angles_deg": ";".join(f"{x:.3f}" for x in hoh_angles),
        "PbH_min_A": min(pb_h_distances) if pb_h_distances else "",
        "PbH_max_A": max(pb_h_distances) if pb_h_distances else "",
        "nonwater_H_zrel_min_A": min(pb_h_zrel) if pb_h_zrel else "",
        "nonwater_H_zrel_max_A": max(pb_h_zrel) if pb_h_zrel else "",
        "nonwater_H_nearest_Pb": ";".join(str(x) for x in nearest_pb_indices),
        "nonwater_H_same_Pb": (
            len(set(nearest_pb_indices)) == 1
            if nearest_pb_indices
            else ""
        ),
        "min_HH_A": min(hh_distances) if hh_distances else "",
        "H2_like": (
            min(hh_distances) < H2_MAX_A
            if hh_distances
            else False
        ),
        "min_nonwaterH_waterH_A": (
            min(nonwater_to_water_h)
            if nonwater_to_water_h
            else ""
        ),
        "min_nonwaterH_O_A": (
            min(nonwater_to_o)
            if nonwater_to_o
            else ""
        ),
        "proton_well_like_nonwater_H": proton_well_like,
        "directory": str(job_dir),
    }

    return row


def main():
    harvest = load_harvest()
    rows = []

    for metadata_path in ROOT.rglob("metadata.json"):
        metadata = json.loads(metadata_path.read_text())

        if metadata.get("family") != "proton_well_rescue":
            continue

        row = analyze_job(
            metadata_path.parent,
            metadata,
            harvest,
        )

        if row is not None:
            rows.append(row)

    rows.sort(
        key=lambda row: (
            float(row["target_mu_Ha"])
            if row["target_mu_Ha"] != ""
            else 999.0,
            row["job_id"],
        )
    )

    if not rows:
        raise RuntimeError("No explicit-water proton_well_rescue jobs found.")

    fieldnames = list(rows[0].keys())

    with OUTPUT.open("w", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
        )

        writer.writeheader()
        writer.writerows(rows)

    print("Explicit-water Pb-H states")
    print("==========================")

    for row in rows:
        print(
            f"{row['job_id']:>4}  "
            f"mu={float(row['target_mu_Ha']): .5f}  "
            f"status={row['status']:<20}  "
            f"water={row['water_H_counts_per_O']:<5}  "
            f"OH={row['OH_min_A']:.3f}-{row['OH_max_A']:.3f} A  "
            f"PbH={row['PbH_min_A']:.3f}-{row['PbH_max_A']:.3f} A  "
            f"H2={row['H2_like']}  "
            f"proton_well={row['proton_well_like_nonwater_H']}"
        )

    print()
    print(f"Wrote: {OUTPUT}")


if __name__ == "__main__":
    main()
'''


SCRIPT_PATH.write_text(SCRIPT)
print(f"Wrote: {SCRIPT_PATH}")

Wrote: /pscratch/sd/a/awasthi/Pb_LAR/analysis/analyze_explicit_water_states.py


In [5]:
# Wave 9: explicit-water detached PbH2 pilot.
#
# Scientific purpose:
#   Construct detached PbH2 + 2H2O states with the same hydride-oriented
#   first-shell water motif that successfully rescued the strongly lifted
#   surface Pb(H)2 precursor.
#
# Inputs:
#   X06: detached PbH2 + vacancy at target_mu = -0.11930 Ha
#   X14: detached PbH2 + vacancy at target_mu = -0.09725 Ha
#
# Outputs:
#   X28: detached PbH2 + 2H2O at -0.11930 Ha
#   X29: detached PbH2 + 2H2O at -0.09725 Ha
#
# No existing files or directories are overwritten.

from pathlib import Path
import json
import re

import numpy as np
from ase import Atoms
from ase.io import read, write


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")
CONTROL = ROOT.parent / "_control"

GO_TEMPLATE = CONTROL / "go.py"
SUBMIT_TEMPLATE = CONTROL / "submit_go.sh"

MANIFEST_PATH = ROOT / "analysis/manifests/wave9_detached_PbH2_explicit_water_pilot.json"

OH_BOND_A = 0.970
HOH_ANGLE_DEG = 104.5
HYDRIDE_HH_CONTACT_A = 1.75
GO_MAXSTEP_A = 0.03


JOBS = [
    {
        "job_id": "X28",
        "source_job_id": "X06",
        "source_relpath": "surfaces/Pb111/PbH2_extraction/fixed_mu/m0p1193/Pb_lift_6p0",
        "target_mu_Ha": -0.11930,
        "approx_U_RHE_V_at_pH7": -1.0,
        "job_name": "detH2O_mu1193",
        "relpath": "surfaces/Pb111/PbH2_extraction/explicit_water/detached/m0p1193",
    },
    {
        "job_id": "X29",
        "source_job_id": "X14",
        "source_relpath": "surfaces/Pb111/PbH2_extraction/potential_sweep/detached/m0p09725",
        "target_mu_Ha": -0.09725,
        "approx_U_RHE_V_at_pH7": -1.6,
        "job_name": "detH2O_mu09725",
        "relpath": "surfaces/Pb111/PbH2_extraction/explicit_water/detached/m0p09725",
    },
]


def unit(vector):
    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)

    if norm < 1e-10:
        raise ValueError("Cannot normalize near-zero vector.")

    return vector / norm


def perpendicular_component(vector, axis):
    vector = np.asarray(vector, dtype=float)
    axis = unit(axis)

    result = vector - np.dot(vector, axis) * axis

    if np.linalg.norm(result) > 1e-6:
        return unit(result)

    candidates = [
        np.array([1.0, 0.0, 0.0]),
        np.array([0.0, 1.0, 0.0]),
        np.array([0.0, 0.0, 1.0]),
    ]

    candidate = min(candidates, key=lambda v: abs(np.dot(v, axis)))
    return unit(candidate - np.dot(candidate, axis) * axis)


def identify_detached_pbh2(atoms):
    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    h_indices = np.where(symbols == "H")[0]

    if len(h_indices) != 2:
        raise ValueError(
            f"Detached PbH2 parent should contain exactly 2 H; found {len(h_indices)}."
        )

    nearest_pb = []
    pb_h_distances = []

    for h_idx in h_indices:
        distances = np.array([
            atoms.get_distance(int(pb_idx), int(h_idx), mic=True)
            for pb_idx in pb_indices
        ])

        local = int(np.argmin(distances))

        nearest_pb.append(int(pb_indices[local]))
        pb_h_distances.append(float(distances[local]))

    if nearest_pb[0] != nearest_pb[1]:
        raise ValueError(
            f"The two H atoms do not share one Pb: {nearest_pb}"
        )

    if max(pb_h_distances) > 2.25:
        raise ValueError(
            f"Parent does not look like intact PbH2: Pb-H = {pb_h_distances}"
        )

    hh = atoms.get_distance(
        int(h_indices[0]),
        int(h_indices[1]),
        mic=True,
    )

    if hh < 1.2:
        raise ValueError(
            f"Parent looks H2-like rather than PbH2-like: H-H = {hh:.3f} A"
        )

    return nearest_pb[0], [int(i) for i in h_indices], pb_h_distances, float(hh)


def local_geometry_vectors(atoms, pb_idx, h_indices, i):
    h_idx = h_indices[i]
    other_idx = h_indices[1 - i]

    pb_to_h = atoms.get_distance(
        pb_idx,
        h_idx,
        mic=True,
        vector=True,
    )

    other_to_h = atoms.get_distance(
        other_idx,
        h_idx,
        mic=True,
        vector=True,
    )

    outward = unit(pb_to_h)
    sideways = perpendicular_component(other_to_h, outward)

    return outward, sideways


def make_hydride_oriented_water(target_position, outward, sideways):
    theta = np.deg2rad(HOH_ANGLE_DEG)

    donor_h = target_position + HYDRIDE_HH_CONTACT_A * outward
    oxygen = donor_h + OH_BOND_A * outward

    second_direction = (
        np.cos(theta) * (-outward)
        + np.sin(theta) * sideways
    )

    second_h = oxygen + OH_BOND_A * unit(second_direction)

    return Atoms(
        symbols=["O", "H", "H"],
        positions=[oxygen, donor_h, second_h],
    )


def add_two_waters(parent, pb_idx, h_indices):
    atoms = parent.copy()

    for i, h_idx in enumerate(h_indices):
        outward, sideways = local_geometry_vectors(
            parent,
            pb_idx,
            h_indices,
            i,
        )

        target = parent.positions[h_idx].copy()

        water = make_hydride_oriented_water(
            target,
            outward,
            sideways,
        )

        water.set_cell(parent.cell)
        water.set_pbc(parent.pbc)

        atoms += water

    atoms.set_pbc((True, True, False))

    return atoms


def validate_geometry(atoms, pb_idx, hydride_indices):
    symbols = np.array(atoms.get_chemical_symbols())

    n_pb = int(np.sum(symbols == "Pb"))
    n_o = int(np.sum(symbols == "O"))
    n_h = int(np.sum(symbols == "H"))

    if (n_pb, n_o, n_h) != (36, 2, 6):
        raise ValueError(
            f"Unexpected composition Pb/O/H = {n_pb}/{n_o}/{n_h}; expected 36/2/6."
        )

    # Before go.py performs any species sorting:
    # original system = 36 Pb + 2 H
    # added water atoms = O,H,H,O,H,H
    water_groups = [
        [38, 39, 40],
        [41, 42, 43],
    ]

    contacts = []

    for i, group in enumerate(water_groups):
        o_idx, wh1_idx, wh2_idx = group
        target_h = hydride_indices[i]

        oh1 = atoms.get_distance(o_idx, wh1_idx, mic=True)
        oh2 = atoms.get_distance(o_idx, wh2_idx, mic=True)

        if not (0.94 <= oh1 <= 1.00 and 0.94 <= oh2 <= 1.00):
            raise ValueError(
                f"Water {i + 1} has bad initial O-H distances: "
                f"{oh1:.3f}, {oh2:.3f} A"
            )

        contact = min(
            atoms.get_distance(target_h, wh1_idx, mic=True),
            atoms.get_distance(target_h, wh2_idx, mic=True),
        )

        contacts.append(float(contact))

    pb_h = [
        float(atoms.get_distance(pb_idx, h_idx, mic=True))
        for h_idx in hydride_indices
    ]

    hh = float(
        atoms.get_distance(
            hydride_indices[0],
            hydride_indices[1],
            mic=True,
        )
    )

    max_z = float(np.max(atoms.positions[:, 2]))
    top_gap = float(atoms.cell[2, 2] - max_z)

    if top_gap < 3.0:
        raise ValueError(
            f"Only {top_gap:.2f} A remains above explicit atoms."
        )

    return {
        "PbH_distances_A": pb_h,
        "HH_distance_A": hh,
        "water_H_to_hydride_contacts_A": contacts,
        "top_vacuum_gap_A": top_gap,
    }


def make_go_text(template, target_mu):
    text = template

    if re.search(r"(?m)^\s*target-mu\s+", text):
        raise RuntimeError(
            "Control go.py already contains target-mu; refusing ambiguous edit."
        )

    fluid_marker = "fluid-anion F- 0.5\n"

    if text.count(fluid_marker) != 1:
        raise RuntimeError(
            "Could not uniquely locate fluid-anion line."
        )

    text = text.replace(
        fluid_marker,
        fluid_marker + f"target-mu {target_mu:.5f}\n",
        1,
    )

    cavity_marker = "dump Ionic BoundCharge VfluidTot FluidDensity\n"

    if text.count(cavity_marker) == 1:
        text = text.replace(
            cavity_marker,
            cavity_marker + "dump Ionic Vcavity\n",
            1,
        )

    text, n = re.subn(
        r"maxstep\s*=\s*[0-9.]+",
        f"maxstep={GO_MAXSTEP_A:.2f}",
        text,
        count=1,
    )

    if n != 1:
        raise RuntimeError("Could not uniquely replace FIRE maxstep.")

    return text


def make_submit_text(template, job_name):
    text, n = re.subn(
        r"(?m)^#SBATCH --job-name=.*$",
        f"#SBATCH --job-name={job_name}",
        template,
        count=1,
    )

    if n != 1:
        raise RuntimeError("Could not uniquely replace Slurm job name.")

    return text


# ---------------------------------------------------------------------
# Preflight
# ---------------------------------------------------------------------

for path in [GO_TEMPLATE, SUBMIT_TEMPLATE]:
    if not path.is_file():
        raise FileNotFoundError(path)

if MANIFEST_PATH.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing manifest: {MANIFEST_PATH}"
    )

for job in JOBS:
    source_dir = ROOT / job["source_relpath"]
    target_dir = ROOT / job["relpath"]

    if not (source_dir / "CONTCAR").is_file():
        raise FileNotFoundError(source_dir / "CONTCAR")

    if target_dir.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing directory: {target_dir}"
        )


# ---------------------------------------------------------------------
# Create jobs
# ---------------------------------------------------------------------

go_template = GO_TEMPLATE.read_text()
submit_template = SUBMIT_TEMPLATE.read_text()

manifest_jobs = []

print("Wave 9 detached PbH2 + explicit-water setup")
print("===========================================")

for job in JOBS:
    source_dir = ROOT / job["source_relpath"]
    target_dir = ROOT / job["relpath"]

    parent = read(source_dir / "CONTCAR", format="vasp")
    parent.set_pbc((True, True, False))

    pb_idx, hydride_indices, parent_pb_h, parent_hh = identify_detached_pbh2(parent)

    atoms = add_two_waters(
        parent,
        pb_idx,
        hydride_indices,
    )

    metrics = validate_geometry(
        atoms,
        pb_idx,
        hydride_indices,
    )

    target_dir.mkdir(parents=True)

    write(
        target_dir / "POSCAR",
        atoms,
        format="vasp",
        direct=True,
        vasp5=True,
    )

    write(
        target_dir / "initial.xyz",
        atoms,
        format="extxyz",
    )

    (target_dir / "go.py").write_text(
        make_go_text(
            go_template,
            job["target_mu_Ha"],
        )
    )

    (target_dir / "submit.sh").write_text(
        make_submit_text(
            submit_template,
            job["job_name"],
        )
    )

    metadata = {
        "job_id": job["job_id"],
        "family": "proton_well_rescue",
        "state": "detached_PbH2_plus_vacancy_plus_2H2O",
        "purpose": (
            "Construct a detached PbH2 + vacancy product with two "
            "hydride-oriented explicit first-shell water molecules so that "
            "PbH2 detachment can be compared against the solvated surface "
            "Pb(H)2 precursor at identical stoichiometry."
        ),
        "source_job_id": job["source_job_id"],
        "source_path": job["source_relpath"],
        "target_mu_Ha": job["target_mu_Ha"],
        "approx_U_RHE_V_at_pH7": job["approx_U_RHE_V_at_pH7"],
        "explicit_water_count": 2,
        "water_orientation": "hydride_oriented",
        "construction": {
            "OH_bond_A": OH_BOND_A,
            "HOH_angle_deg": HOH_ANGLE_DEG,
            "hydride_Hwater_contact_A": HYDRIDE_HH_CONTACT_A,
            "common_Pb_index_before_go_sort": pb_idx,
            "parent_H_indices_before_go_sort": hydride_indices,
        },
        "initial_geometry_metrics": metrics,
        "fmax_threshold_eV_A": 0.04,
        "fire_maxstep_A": GO_MAXSTEP_A,
        "potential_mapping_note": (
            "approx_U_RHE_V_at_pH7 is historical project calibration; "
            "target_mu_Ha is authoritative."
        ),
    }

    (target_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n"
    )

    manifest_jobs.append(
        {
            "job_id": job["job_id"],
            "directory": job["relpath"],
            "source_job_id": job["source_job_id"],
            "target_mu_Ha": job["target_mu_Ha"],
        }
    )

    print()
    print(
        f"{job['job_id']}  mu={job['target_mu_Ha']:.5f}  "
        f"source={job['source_job_id']}"
    )
    print(
        "Pb-H:              "
        + ", ".join(
            f"{x:.4f} A"
            for x in metrics["PbH_distances_A"]
        )
    )
    print(
        f"H-H:               "
        f"{metrics['HH_distance_A']:.4f} A"
    )
    print(
        "water contacts:    "
        + ", ".join(
            f"{x:.4f} A"
            for x in metrics["water_H_to_hydride_contacts_A"]
        )
    )
    print(
        f"top vacuum gap:     "
        f"{metrics['top_vacuum_gap_A']:.4f} A"
    )


# ---------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------

manifest = {
    "wave": 9,
    "name": "detached PbH2 explicit-water pilot",
    "scientific_question": (
        "Can the detached PbH2 + vacancy product be stabilized with the same "
        "two-water first-shell motif as the solvated surface Pb(H)2 precursor, "
        "allowing balanced fixed-mu detachment thermodynamics?"
    ),
    "jobs": manifest_jobs,
}

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2) + "\n"
)

print()
print(f"Manifest: {MANIFEST_PATH}")

Wave 9 detached PbH2 + explicit-water setup

X28  mu=-0.11930  source=X06
Pb-H:              1.9031 A, 1.9031 A
H-H:               2.6069 A
water contacts:    1.7500 A, 1.7500 A
top vacuum gap:     5.5849 A

X29  mu=-0.09725  source=X14
Pb-H:              1.9298 A, 1.9298 A
H-H:               2.6120 A
water contacts:    1.7500 A, 1.7500 A
top vacuum gap:     5.5570 A

Manifest: /pscratch/sd/a/awasthi/Pb_LAR/analysis/manifests/wave9_detached_PbH2_explicit_water_pilot.json


In [6]:
from pathlib import Path
import csv
import re

import matplotlib.pyplot as plt
import numpy as np
from ase.io import read


ROOT = Path("/pscratch/sd/a/awasthi/Pb_LAR")

JOBS = {
    "X10_dry_proton_well": ROOT / "surfaces/Pb111/PbH2_extraction/potential_sweep/precursor/m0p09725",
    "X22_hydrated_rescue": ROOT / "surfaces/Pb111/proton_well_rescue/hydride_oriented/m0p09725",
}

OUTDIR = ROOT / "analysis/proton_well_diagnostics"

EXPECTED_OUTPUTS = [
    OUTDIR / "01_cavity_shape.png",
    OUTDIR / "02_electron_density_with_cavity.png",
    OUTDIR / "03_bound_charge_with_cavity.png",
    OUTDIR / "04_dielectric_response_with_cavity.png",
    OUTDIR / "05_outward_profiles.png",
    OUTDIR / "point_metrics.csv",
    OUTDIR / "outward_profiles.csv",
]

existing = [path for path in EXPECTED_OUTPUTS if path.exists()]
if existing:
    raise FileExistsError(
        "Refusing to overwrite existing proton-well diagnostics:\n"
        + "\n".join(str(path) for path in existing)
    )

OUTDIR.mkdir(parents=True, exist_ok=True)


FIELD_FILES = {
    "n": "job.n",
    "shape": "job.fluidShape",
    "nbound": "job.nbound",
    "rho_diel": "job.fluidRhoDiel",
    "vfluid": "job.V_fluidTot",
}


def unit(vector):
    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)

    if norm < 1e-12:
        raise ValueError("Cannot normalize a near-zero vector.")

    return vector / norm


def cart_to_frac(points, cell):
    points = np.asarray(points, dtype=float)
    return points @ np.linalg.inv(np.asarray(cell))


def parse_fftbox(out_path, n_values):
    candidates = []

    for line in out_path.read_text(errors="replace").splitlines():
        if "Chosen fftbox size" not in line:
            continue

        numbers = [int(x) for x in re.findall(r"\d+", line)]

        for i in range(max(0, len(numbers) - 2)):
            shape = tuple(numbers[i:i + 3])

            if len(shape) == 3 and np.prod(shape) == n_values:
                candidates.append(shape)

    candidates = list(dict.fromkeys(candidates))

    if len(candidates) != 1:
        relevant = [
            line
            for line in out_path.read_text(errors="replace").splitlines()
            if "Chosen fftbox size" in line
        ]

        raise RuntimeError(
            f"Could not uniquely identify FFT grid from {out_path}.\n"
            f"Electron-density values: {n_values}\n"
            f"Candidates: {candidates}\n"
            + "\n".join(relevant)
        )

    return candidates[0]


def load_raw_field(path, normal_shape):
    raw = np.fromfile(path, dtype="<f8")
    n_normal = int(np.prod(normal_shape))

    if raw.size == n_normal:
        return raw.reshape(normal_shape), "normal"

    doubled_z_shape = (
        normal_shape[0],
        normal_shape[1],
        2 * normal_shape[2],
    )

    if raw.size == 2 * n_normal:
        return raw.reshape(doubled_z_shape), "double_z"

    raise ValueError(
        f"{path} contains {raw.size} doubles; expected "
        f"{n_normal} or {2 * n_normal}."
    )


def trilinear(array, coordinates):
    coordinates = np.asarray(coordinates, dtype=float)

    i0 = np.floor(coordinates).astype(int)
    frac = coordinates - np.floor(coordinates)

    i1 = i0 + 1

    for axis, size in enumerate(array.shape):
        i0[:, axis] %= size
        i1[:, axis] %= size

    x, y, z = frac[:, 0], frac[:, 1], frac[:, 2]

    c000 = array[i0[:, 0], i0[:, 1], i0[:, 2]]
    c001 = array[i0[:, 0], i0[:, 1], i1[:, 2]]
    c010 = array[i0[:, 0], i1[:, 1], i0[:, 2]]
    c011 = array[i0[:, 0], i1[:, 1], i1[:, 2]]
    c100 = array[i1[:, 0], i0[:, 1], i0[:, 2]]
    c101 = array[i1[:, 0], i0[:, 1], i1[:, 2]]
    c110 = array[i1[:, 0], i1[:, 1], i0[:, 2]]
    c111 = array[i1[:, 0], i1[:, 1], i1[:, 2]]

    c00 = c000 * (1.0 - z) + c001 * z
    c01 = c010 * (1.0 - z) + c011 * z
    c10 = c100 * (1.0 - z) + c101 * z
    c11 = c110 * (1.0 - z) + c111 * z

    c0 = c00 * (1.0 - y) + c01 * y
    c1 = c10 * (1.0 - y) + c11 * y

    return c0 * (1.0 - x) + c1 * x


def identify_relevant_h(atoms):
    symbols = np.array(atoms.get_chemical_symbols())

    pb_indices = np.where(symbols == "Pb")[0]
    o_indices = np.where(symbols == "O")[0]
    h_indices = np.where(symbols == "H")[0]

    water_h = set()

    if len(o_indices):
        for h_idx in h_indices:
            nearest_o = min(
                atoms.get_distance(int(h_idx), int(o_idx), mic=True)
                for o_idx in o_indices
            )

            if nearest_o <= 1.25:
                water_h.add(int(h_idx))

    active_h = [
        int(h_idx)
        for h_idx in h_indices
        if int(h_idx) not in water_h
    ]

    if len(active_h) != 2:
        raise ValueError(
            f"Expected exactly two non-water H atoms; found {active_h}."
        )

    nearest_pb = []

    for h_idx in active_h:
        distances = [
            atoms.get_distance(h_idx, int(pb_idx), mic=True)
            for pb_idx in pb_indices
        ]

        nearest_pb.append(int(pb_indices[int(np.argmin(distances))]))

    if nearest_pb[0] != nearest_pb[1]:
        raise ValueError(
            f"The two relevant H atoms do not share one Pb: {nearest_pb}"
        )

    return nearest_pb[0], active_h, water_h


def sample_fractional_on_fluid(array, frac, normal_shape, z_sign, z_offset):
    frac = np.asarray(frac, dtype=float).copy()

    frac[:, 0] %= 1.0
    frac[:, 1] %= 1.0

    coords = np.empty_like(frac)

    coords[:, 0] = frac[:, 0] * normal_shape[0]
    coords[:, 1] = frac[:, 1] * normal_shape[1]
    coords[:, 2] = z_sign * frac[:, 2] * normal_shape[2] + z_offset

    return trilinear(array, coords)


def infer_fluid_z_mapping(atoms, shape_field, normal_shape, central_pb):
    if shape_field.shape == tuple(normal_shape):
        return 1.0, 0.0, {
            "mean_shape_explicit_atoms": np.nan,
            "mean_shape_vacuum": np.nan,
        }

    expected = (
        normal_shape[0],
        normal_shape[1],
        2 * normal_shape[2],
    )

    if shape_field.shape != expected:
        raise ValueError(
            f"Unexpected fluid grid {shape_field.shape}; expected {expected}."
        )

    symbols = np.array(atoms.get_chemical_symbols())

    low_indices = [
        i
        for i, symbol in enumerate(symbols)
        if symbol in {"Pb", "O"}
    ]

    low_cart = atoms.positions[low_indices]
    low_frac = cart_to_frac(low_cart, atoms.cell)

    pb_indices = np.where(symbols == "Pb")[0]
    pb_z = atoms.positions[pb_indices, 2]

    top_z = float(np.max(pb_z))
    bottom_z = float(np.min(pb_z))
    cell_z = float(atoms.cell[2, 2])

    vacuum_top_z = min(top_z + 6.0, cell_z - 1.0)
    vacuum_bottom_z = max(bottom_z - 6.0, 1.0)

    pb_xy = atoms.positions[central_pb, :2]

    vacuum_cart = np.array([
        [pb_xy[0], pb_xy[1], vacuum_top_z],
        [pb_xy[0], pb_xy[1], vacuum_bottom_z],
    ])

    vacuum_frac = cart_to_frac(vacuum_cart, atoms.cell)

    best = None

    def evaluate(sign, offset):
        low_values = sample_fractional_on_fluid(
            shape_field,
            low_frac,
            normal_shape,
            sign,
            offset,
        )

        vacuum_values = sample_fractional_on_fluid(
            shape_field,
            vacuum_frac,
            normal_shape,
            sign,
            offset,
        )

        score = (
            np.mean(low_values ** 2)
            + 0.5 * np.mean((vacuum_values - 1.0) ** 2)
        )

        return float(score), low_values, vacuum_values

    # Coarse search over the entire doubled z grid.
    for sign in (1.0, -1.0):
        for offset in np.arange(0.0, 2.0 * normal_shape[2], 1.0):
            score, low_values, vacuum_values = evaluate(sign, offset)

            if best is None or score < best[0]:
                best = (
                    score,
                    sign,
                    float(offset),
                    low_values,
                    vacuum_values,
                )

    # Refine to sub-grid accuracy around the best integer shift.
    _, best_sign, best_offset, _, _ = best

    for offset in np.arange(best_offset - 1.0, best_offset + 1.0001, 0.05):
        score, low_values, vacuum_values = evaluate(best_sign, offset)

        if score < best[0]:
            best = (
                score,
                best_sign,
                float(offset),
                low_values,
                vacuum_values,
            )

    score, sign, offset, low_values, vacuum_values = best

    diagnostics = {
        "score": score,
        "mean_shape_explicit_atoms": float(np.mean(low_values)),
        "max_shape_explicit_atoms": float(np.max(low_values)),
        "mean_shape_vacuum": float(np.mean(vacuum_values)),
        "vacuum_values": vacuum_values.tolist(),
    }

    if diagnostics["mean_shape_explicit_atoms"] > 0.25:
        raise RuntimeError(
            "Could not find a convincing fluid-grid alignment: "
            f"mean shape at Pb/O = {diagnostics['mean_shape_explicit_atoms']:.3f}"
        )

    if diagnostics["mean_shape_vacuum"] < 0.70:
        raise RuntimeError(
            "Could not find a convincing fluid-grid alignment: "
            f"mean vacuum shape = {diagnostics['mean_shape_vacuum']:.3f}"
        )

    return sign, offset, diagnostics


def load_job(name, directory):
    atoms = read(directory / "CONTCAR", format="vasp")
    atoms.set_pbc((True, True, False))

    n_path = directory / FIELD_FILES["n"]

    if not n_path.is_file():
        raise FileNotFoundError(n_path)

    n_values = n_path.stat().st_size // 8
    normal_shape = parse_fftbox(directory / "out", n_values)

    fields = {}
    field_modes = {}

    for key, filename in FIELD_FILES.items():
        path = directory / filename

        if not path.is_file():
            raise FileNotFoundError(path)

        fields[key], field_modes[key] = load_raw_field(
            path,
            normal_shape,
        )

    central_pb, active_h, water_h = identify_relevant_h(atoms)

    fluid_sign, fluid_offset, fluid_diagnostics = infer_fluid_z_mapping(
        atoms,
        fields["shape"],
        normal_shape,
        central_pb,
    )

    return {
        "name": name,
        "directory": directory,
        "atoms": atoms,
        "normal_shape": normal_shape,
        "fields": fields,
        "field_modes": field_modes,
        "central_pb": central_pb,
        "active_h": active_h,
        "water_h": water_h,
        "fluid_sign": fluid_sign,
        "fluid_offset": fluid_offset,
        "fluid_diagnostics": fluid_diagnostics,
    }


def sample_job_field(job, field_key, cart_points):
    cart_points = np.asarray(cart_points, dtype=float)
    original_shape = cart_points.shape[:-1]

    points = cart_points.reshape(-1, 3)

    frac = cart_to_frac(
        points,
        job["atoms"].cell,
    )

    field = job["fields"][field_key]
    mode = job["field_modes"][field_key]
    normal_shape = job["normal_shape"]

    frac[:, 0] %= 1.0
    frac[:, 1] %= 1.0
    frac[:, 2] %= 1.0

    if mode == "normal":
        coords = frac * np.asarray(normal_shape, dtype=float)
        values = trilinear(field, coords)

    elif mode == "double_z":
        values = sample_fractional_on_fluid(
            field,
            frac,
            normal_shape,
            job["fluid_sign"],
            job["fluid_offset"],
        )

    else:
        raise ValueError(mode)

    return values.reshape(original_shape)


def minimum_image_vector(origin, position, cell):
    frac = cart_to_frac(
        np.asarray(position) - np.asarray(origin),
        cell,
    )

    frac[0] -= np.round(frac[0])
    frac[1] -= np.round(frac[1])

    return frac @ np.asarray(cell)


def make_plane_basis(job):
    atoms = job["atoms"]
    pb_idx = job["central_pb"]
    h1, h2 = job["active_h"]

    origin = atoms.positions[pb_idx].copy()

    v1 = atoms.get_distance(
        pb_idx,
        h1,
        mic=True,
        vector=True,
    )

    v2 = atoms.get_distance(
        pb_idx,
        h2,
        mic=True,
        vector=True,
    )

    e_u = unit(v2 - v1)

    midpoint_vector = 0.5 * (v1 + v2)
    e_v_raw = midpoint_vector - np.dot(midpoint_vector, e_u) * e_u
    e_v = unit(e_v_raw)

    if e_v[2] < 0.0:
        e_v *= -1.0

    e_normal = unit(np.cross(e_u, e_v))

    return origin, e_u, e_v, e_normal


def build_plane(job):
    origin, e_u, e_v, e_normal = make_plane_basis(job)

    u = np.linspace(-4.5, 4.5, 361)
    v = np.linspace(-2.5, 7.0, 381)

    U, V = np.meshgrid(
        u,
        v,
        indexing="xy",
    )

    points = (
        origin[None, None, :]
        + U[:, :, None] * e_u[None, None, :]
        + V[:, :, None] * e_v[None, None, :]
    )

    slices = {}

    for key in FIELD_FILES:
        slices[key] = sample_job_field(
            job,
            key,
            points,
        )

    return {
        "origin": origin,
        "e_u": e_u,
        "e_v": e_v,
        "e_normal": e_normal,
        "u": u,
        "v": v,
        "U": U,
        "V": V,
        "slices": slices,
    }


def overlay_atoms(ax, job, plane):
    atoms = job["atoms"]
    origin = plane["origin"]
    e_u = plane["e_u"]
    e_v = plane["e_v"]
    e_normal = plane["e_normal"]

    projected = []

    for idx, (symbol, position) in enumerate(
        zip(atoms.get_chemical_symbols(), atoms.positions)
    ):
        dr = minimum_image_vector(
            origin,
            position,
            atoms.cell,
        )

        u = float(np.dot(dr, e_u))
        v = float(np.dot(dr, e_v))
        out_of_plane = abs(float(np.dot(dr, e_normal)))

        if out_of_plane > 0.75:
            continue

        if not (-4.8 <= u <= 4.8 and -2.8 <= v <= 7.3):
            continue

        projected.append((idx, symbol, u, v))

    styles = {
        "Pb": dict(s=42, marker="o", facecolor="0.65", edgecolor="0.25"),
        "O": dict(s=52, marker="o", facecolor="tab:red", edgecolor="black"),
        "H": dict(s=38, marker="o", facecolor="white", edgecolor="black"),
    }

    for symbol in ["Pb", "O", "H"]:
        subset = [
            item
            for item in projected
            if item[1] == symbol
        ]

        if not subset:
            continue

        ax.scatter(
            [item[2] for item in subset],
            [item[3] for item in subset],
            linewidths=0.6,
            zorder=6,
            **styles[symbol],
        )

    # Highlight the Pb atom shared by the two relevant H atoms.
    for idx, symbol, u, v in projected:
        if idx == job["central_pb"]:
            ax.scatter(
                [u],
                [v],
                s=90,
                marker="s",
                facecolor="none",
                edgecolor="black",
                linewidths=1.4,
                zorder=7,
            )

        if idx in job["active_h"]:
            ax.scatter(
                [u],
                [v],
                s=90,
                marker="*",
                facecolor="none",
                edgecolor="black",
                linewidths=1.2,
                zorder=7,
            )


def plot_pair(jobs, planes, field_key, filename, label, transform=None, symmetric=False):
    arrays = []

    for job in jobs:
        data = planes[job["name"]]["slices"][field_key]

        if transform is not None:
            data = transform(data)

        arrays.append(data)

    if symmetric:
        concatenated = np.concatenate(
            [np.abs(array[np.isfinite(array)]).ravel() for array in arrays]
        )

        vmax = float(np.quantile(concatenated, 0.995))

        if vmax <= 0.0:
            vmax = 1.0

        vmin = -vmax
        cmap = "RdBu_r"

    else:
        concatenated = np.concatenate(
            [array[np.isfinite(array)].ravel() for array in arrays]
        )

        vmin = float(np.quantile(concatenated, 0.01))
        vmax = float(np.quantile(concatenated, 0.995))
        cmap = "viridis"

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5.2),
        constrained_layout=True,
    )

    image = None

    for ax, job, data in zip(axes, jobs, arrays):
        plane = planes[job["name"]]

        image = ax.imshow(
            data,
            origin="lower",
            extent=[
                plane["u"][0],
                plane["u"][-1],
                plane["v"][0],
                plane["v"][-1],
            ],
            aspect="equal",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )

        if field_key != "shape":
            ax.contour(
                plane["U"],
                plane["V"],
                plane["slices"]["shape"],
                levels=[0.5],
                linewidths=[1.5],
                colors=["black"],
            )

        overlay_atoms(
            ax,
            job,
            plane,
        )

        ax.set_title(job["name"])
        ax.set_xlabel("H-H direction / Å")
        ax.set_ylabel("Pb → H midpoint direction / Å")

    fig.colorbar(
        image,
        ax=axes,
        label=label,
        shrink=0.88,
    )

    fig.savefig(
        OUTDIR / filename,
        dpi=220,
    )

    plt.close(fig)


jobs = [
    load_job(name, directory)
    for name, directory in JOBS.items()
]

planes = {
    job["name"]: build_plane(job)
    for job in jobs
}


# ---------------------------------------------------------------------
# Figure 1: the cavity itself
# shape = 0 -> solute/cavity, shape = 1 -> bulk fluid
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5.2),
    constrained_layout=True,
)

image = None

for ax, job in zip(axes, jobs):
    plane = planes[job["name"]]
    shape = plane["slices"]["shape"]

    image = ax.imshow(
        shape,
        origin="lower",
        extent=[
            plane["u"][0],
            plane["u"][-1],
            plane["v"][0],
            plane["v"][-1],
        ],
        aspect="equal",
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
    )

    ax.contour(
        plane["U"],
        plane["V"],
        shape,
        levels=[0.1, 0.5, 0.9],
        colors=["black", "black", "black"],
        linewidths=[0.6, 1.6, 0.6],
        linestyles=["dotted", "solid", "dashed"],
    )

    overlay_atoms(
        ax,
        job,
        plane,
    )

    ax.set_title(job["name"])
    ax.set_xlabel("H-H direction / Å")
    ax.set_ylabel("Pb → H midpoint direction / Å")

fig.colorbar(
    image,
    ax=axes,
    label="fluidShape: 0 = solute/cavity, 1 = bulk fluid",
    shrink=0.88,
)

fig.savefig(
    OUTDIR / "01_cavity_shape.png",
    dpi=220,
)

plt.close(fig)


# ---------------------------------------------------------------------
# Figure 2: explicit-system electron density + s=0.5 cavity boundary
# ---------------------------------------------------------------------

plot_pair(
    jobs,
    planes,
    field_key="n",
    filename="02_electron_density_with_cavity.png",
    label=r"log10 electron density [e / bohr$^3$]",
    transform=lambda x: np.log10(np.clip(x, 1e-10, None)),
    symmetric=False,
)


# ---------------------------------------------------------------------
# Figure 3: total fluid bound charge + cavity boundary
# ---------------------------------------------------------------------

plot_pair(
    jobs,
    planes,
    field_key="nbound",
    filename="03_bound_charge_with_cavity.png",
    label="BoundCharge [JDFTx raw density units]",
    symmetric=True,
)


# ---------------------------------------------------------------------
# Figure 4: dielectric component of fluid bound charge
# ---------------------------------------------------------------------

plot_pair(
    jobs,
    planes,
    field_key="rho_diel",
    filename="04_dielectric_response_with_cavity.png",
    label="fluidRhoDiel [JDFTx raw density units]",
    symmetric=True,
)


# ---------------------------------------------------------------------
# Point metrics and outward profiles
# ---------------------------------------------------------------------

metric_rows = []
profile_rows = []
profiles_for_plot = {}

for job in jobs:
    atoms = job["atoms"]
    pb_idx = job["central_pb"]

    individual_profiles = []

    for local_h_number, h_idx in enumerate(job["active_h"], start=1):
        h_position = atoms.positions[h_idx].copy()

        pb_to_h = atoms.get_distance(
            pb_idx,
            h_idx,
            mic=True,
            vector=True,
        )

        pb_h_distance = float(np.linalg.norm(pb_to_h))
        outward = unit(pb_to_h)

        point = h_position[None, :]

        values_at_h = {
            key: float(sample_job_field(job, key, point)[0])
            for key in FIELD_FILES
        }

        t = np.linspace(
            0.0,
            4.0,
            401,
        )

        line_points = (
            h_position[None, :]
            + t[:, None] * outward[None, :]
        )

        line = {
            key: sample_job_field(
                job,
                key,
                line_points,
            )
            for key in FIELD_FILES
        }

        shape_line = line["shape"]

        if shape_line[0] >= 0.5:
            cavity_clearance = 0.0

        else:
            crossings = np.where(
                (shape_line[:-1] < 0.5)
                & (shape_line[1:] >= 0.5)
            )[0]

            if len(crossings):
                i = int(crossings[0])

                x0, x1 = t[i], t[i + 1]
                y0, y1 = shape_line[i], shape_line[i + 1]

                cavity_clearance = float(
                    x0 + (0.5 - y0) * (x1 - x0) / (y1 - y0)
                )

            else:
                cavity_clearance = np.nan

        metric_rows.append(
            {
                "job": job["name"],
                "H_number": local_h_number,
                "H_atom_index": h_idx,
                "Pb_atom_index": pb_idx,
                "PbH_A": pb_h_distance,
                "fluidShape_at_H": values_at_h["shape"],
                "electron_density_at_H_e_bohr3": values_at_h["n"],
                "BoundCharge_at_H": values_at_h["nbound"],
                "fluidRhoDiel_at_H": values_at_h["rho_diel"],
                "VfluidTot_at_H": values_at_h["vfluid"],
                "cavity_clearance_outward_A": cavity_clearance,
            }
        )

        individual_profiles.append(
            {
                "t": t,
                **line,
            }
        )

    average_profile = {
        "t": individual_profiles[0]["t"],
    }

    for key in FIELD_FILES:
        average_profile[key] = np.mean(
            np.stack(
                [profile[key] for profile in individual_profiles],
                axis=0,
            ),
            axis=0,
        )

    profiles_for_plot[job["name"]] = average_profile

    for i, t_value in enumerate(average_profile["t"]):
        profile_rows.append(
            {
                "job": job["name"],
                "distance_outward_from_H_A": float(t_value),
                "fluidShape": float(average_profile["shape"][i]),
                "electron_density_e_bohr3": float(average_profile["n"][i]),
                "BoundCharge": float(average_profile["nbound"][i]),
                "fluidRhoDiel": float(average_profile["rho_diel"][i]),
                "VfluidTot": float(average_profile["vfluid"][i]),
            }
        )


with (OUTDIR / "point_metrics.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(metric_rows[0].keys()),
    )

    writer.writeheader()
    writer.writerows(metric_rows)


with (OUTDIR / "outward_profiles.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=list(profile_rows[0].keys()),
    )

    writer.writeheader()
    writer.writerows(profile_rows)


# ---------------------------------------------------------------------
# Figure 5: direct one-dimensional comparison outward from the two H atoms
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4.5),
    constrained_layout=True,
)

for job in jobs:
    profile = profiles_for_plot[job["name"]]

    axes[0].plot(
        profile["t"],
        profile["shape"],
        label=job["name"],
    )

    axes[1].plot(
        profile["t"],
        np.log10(np.clip(profile["n"], 1e-10, None)),
        label=job["name"],
    )

axes[0].axhline(
    0.5,
    linestyle="--",
    linewidth=1.0,
)

axes[0].set_xlabel("Distance outward from H / Å")
axes[0].set_ylabel("fluidShape")
axes[0].set_ylim(-0.05, 1.05)
axes[0].legend()

axes[1].set_xlabel("Distance outward from H / Å")
axes[1].set_ylabel(r"log10 electron density [e / bohr$^3$]")
axes[1].legend()

fig.savefig(
    OUTDIR / "05_outward_profiles.png",
    dpi=220,
)

plt.close(fig)


# ---------------------------------------------------------------------
# Compact report
# ---------------------------------------------------------------------

print("Proton-well field comparison")
print("============================")
print()

for job in jobs:
    diagnostics = job["fluid_diagnostics"]

    print(job["name"])
    print("-" * len(job["name"]))
    print(f"normal FFT grid:       {job['normal_shape']}")
    print(f"fluid field grid:      {job['fields']['shape'].shape}")
    print(f"fluid z sign:          {job['fluid_sign']:+.0f}")
    print(f"fluid z offset:        {job['fluid_offset']:.3f} grid points")
    print(
        f"<shape> at Pb/O:       "
        f"{diagnostics['mean_shape_explicit_atoms']:.5f}"
    )
    print(
        f"max shape at Pb/O:     "
        f"{diagnostics['max_shape_explicit_atoms']:.5f}"
    )
    print(
        f"<shape> in vacuum:     "
        f"{diagnostics['mean_shape_vacuum']:.5f}"
    )

    for row in metric_rows:
        if row["job"] != job["name"]:
            continue

        print(
            f"H{row['H_number']}: "
            f"Pb-H={row['PbH_A']:.3f} A  "
            f"shape(H)={row['fluidShape_at_H']:.4f}  "
            f"n(H)={row['electron_density_at_H_e_bohr3']:.3e}  "
            f"nbound(H)={row['BoundCharge_at_H']:+.3e}  "
            f"rhoDiel(H)={row['fluidRhoDiel_at_H']:+.3e}  "
            f"cavity_out={row['cavity_clearance_outward_A']:.3f} A"
        )

    print()

print(f"Figures and CSVs written to: {OUTDIR}")

Proton-well field comparison

X10_dry_proton_well
-------------------
normal FFT grid:       (96, 96, 336)
fluid field grid:      (96, 96, 672)
fluid z sign:          -1
fluid z offset:        163.950 grid points
<shape> at Pb/O:       -0.00000
max shape at Pb/O:     0.00000
<shape> in vacuum:     1.00000
H1: Pb-H=3.897 A  shape(H)=0.8594  n(H)=7.022e-05  nbound(H)=+3.367e-01  rhoDiel(H)=+3.114e-04  cavity_out=0.000 A
H2: Pb-H=3.897 A  shape(H)=0.8315  n(H)=7.116e-05  nbound(H)=+3.403e-01  rhoDiel(H)=+2.940e-04  cavity_out=0.000 A

X22_hydrated_rescue
-------------------
normal FFT grid:       (96, 96, 336)
fluid field grid:      (96, 96, 672)
fluid z sign:          -1
fluid z offset:        173.750 grid points
<shape> at Pb/O:       0.05646
max shape at Pb/O:     0.52511
<shape> in vacuum:     1.00000
H1: Pb-H=2.068 A  shape(H)=-0.0000  n(H)=2.743e-01  nbound(H)=-1.298e-04  rhoDiel(H)=+7.311e-06  cavity_out=2.597 A
H2: Pb-H=2.068 A  shape(H)=-0.0000  n(H)=2.783e-01  nbound(H)=-1.307e-